# LatentGate · direction-balanced path-control pilot

One seed, one direction-neutral prompt, mirrored left/right box paths, and shared initial noise. Each target gets Base, foreground/background path control, and full DiTraj (early STD-RoPE). The six generated clips are blinded for manual per-frame visibility, identity/count, center, box, and shape labels; red-pixel tracking is secondary only.

This is a small 17-frame/30-step adaptation, not a reproduction of the paper’s 81-frame A100 benchmark. It is a screening experiment, not a claim of improvement. Upstream DiTraj states CC BY-NC 4.0; research use only.

**Closed admission gate:** every setup, preflight, and run cell requires a fresh manual receipt from the designated Kaggle account. The runner cannot query Kaggle billing/quota. The receipt is an operator attestation, not provider-signed proof; the kernel cannot re-hash the staged `.ipynb`, so its SHA must be copied from the reviewed local pin. No install, model download, CUDA check, or generation is allowed without the matching receipt.

Run cells individually. The full 30-step and feedback paths remain opt-in and require a new receipt for their exact arm. Preliminary frames and reports are local diagnostics only: no ZIP/export/scoring until the registered per-frame SHA-256/dimension manifest and separate background-stability measure are present.

In [ ]:
# Manual admission is required before clone, package installation, or model/CUDA work.
# Kaggle does not expose its billing/quota state to this runner; this receipt is a human attestation.
from pathlib import Path
import json, sys

ADMISSION_MODULE_SOURCE = '"""Fail-closed validation for a manually reviewed, no-paid DiTraj admission receipt.\n\nThis module does not query Kaggle billing, quota, or account state. It validates\nthat an operator\'s fresh attestation is complete and matches the intended arm,\nthe locally pinned notebook hash, and the exact runner bytes before any runner\nimports model, CUDA, or upstream-patching code.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nfrom typing import Any\n\n\nRECEIPT_SCHEMA = "latentgate.ditraj_free_admission.v1"\nDESIGNATED_KAGGLE_ACCOUNT = "bvyashwantkumar"\nMAX_RECEIPT_AGE = timedelta(minutes=15)\nMAX_FUTURE_SKEW = timedelta(minutes=2)\nSHA256_RE = re.compile(r"^[0-9a-f]{64}$")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _parse_utc(value: Any) -> datetime:\n    if not isinstance(value, str) or not value.strip():\n        raise ValueError("receipt checked_at_utc must be an ISO-8601 UTC timestamp")\n    normalized = value.strip()\n    if normalized.endswith("Z"):\n        normalized = normalized[:-1] + "+00:00"\n    try:\n        parsed = datetime.fromisoformat(normalized)\n    except ValueError as exc:\n        raise ValueError("receipt checked_at_utc must be an ISO-8601 UTC timestamp") from exc\n    if parsed.tzinfo is None or parsed.utcoffset() != timedelta(0):\n        raise ValueError("receipt checked_at_utc must include UTC timezone information")\n    return parsed.astimezone(timezone.utc)\n\n\ndef validate_admission_receipt(\n    receipt_path: Path,\n    *,\n    intended_arm: str,\n    expected_notebook_sha256: str,\n    runner_path: Path | None = None,\n    expected_runner_sha256: str | None = None,\n    now: datetime | None = None,\n) -> dict[str, Any]:\n    """Validate a manual receipt; returns a JSON-safe, explicit provenance record.\n\n    The notebook file is not available to a Kaggle kernel for independent\n    hashing. Therefore `expected_notebook_sha256` must be transcribed from the\n    reviewed local source pin; the receipt records that operator check. During\n    pre-install notebook setup, the runner hash is compared with its pinned\n    local hash. At CLI invocation, the executing runner file is re-hashed.\n    """\n    if not intended_arm or not intended_arm.strip():\n        raise ValueError("an explicit intended arm is required")\n    if not SHA256_RE.fullmatch(expected_notebook_sha256):\n        raise ValueError("expected notebook SHA-256 must be 64 lowercase hex characters")\n    if not receipt_path.is_file():\n        raise ValueError(f"no admission receipt at {receipt_path}; execution remains blocked")\n    if expected_runner_sha256 is not None and not SHA256_RE.fullmatch(expected_runner_sha256):\n        raise ValueError("expected runner SHA-256 must be 64 lowercase hex characters")\n    if runner_path is None and expected_runner_sha256 is None:\n        raise ValueError("an expected runner hash or runner file is required")\n    if runner_path is not None and not runner_path.is_file():\n        raise ValueError(f"runner source is missing at {runner_path}; execution remains blocked")\n\n    try:\n        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))\n    except (OSError, json.JSONDecodeError) as exc:\n        raise ValueError("admission receipt is unreadable or invalid JSON") from exc\n    if not isinstance(receipt, dict):\n        raise ValueError("admission receipt must be a JSON object")\n\n    required = {\n        "schema",\n        "account",\n        "checked_at_utc",\n        "notebook_sha256",\n        "runner_sha256",\n        "intended_arm",\n        "free_t4_admission_confirmed",\n        "paid_prompt_seen",\n        "admission_provenance",\n        "evidence_sha256",\n    }\n    missing = sorted(required - receipt.keys())\n    if missing:\n        raise ValueError(f"admission receipt is missing required fields: {\', \'.join(missing)}")\n    if receipt["schema"] != RECEIPT_SCHEMA:\n        raise ValueError("admission receipt schema is not supported")\n    if receipt["account"] != DESIGNATED_KAGGLE_ACCOUNT:\n        raise ValueError("receipt account does not match the designated Kaggle account")\n    if receipt["intended_arm"] != intended_arm:\n        raise ValueError("receipt intended_arm does not match this run")\n    if receipt["free_t4_admission_confirmed"] is not True:\n        raise ValueError("fresh free T4 admission is not attested; execution remains blocked")\n    if receipt["paid_prompt_seen"] is not False:\n        raise ValueError("a paid/upgrade prompt was seen or not explicitly ruled out")\n    if receipt["notebook_sha256"] != expected_notebook_sha256:\n        raise ValueError("receipt notebook SHA-256 does not match the reviewed local notebook pin")\n    if not isinstance(receipt["runner_sha256"], str) or not SHA256_RE.fullmatch(receipt["runner_sha256"]):\n        raise ValueError("receipt runner SHA-256 must be 64 lowercase hex characters")\n\n    actual_runner_sha256 = sha256_file(runner_path) if runner_path is not None else expected_runner_sha256\n    if receipt["runner_sha256"] != actual_runner_sha256:\n        raise ValueError("receipt runner SHA-256 does not match the executing runner bytes")\n    if not isinstance(receipt["evidence_sha256"], str) or not SHA256_RE.fullmatch(receipt["evidence_sha256"]):\n        raise ValueError("receipt evidence_sha256 must identify the reviewed admission evidence")\n    provenance = receipt["admission_provenance"]\n    if not isinstance(provenance, str) or len(provenance.strip()) < 12:\n        raise ValueError("receipt admission_provenance must describe the manual UI evidence")\n\n    checked_at = _parse_utc(receipt["checked_at_utc"])\n    now_utc = (now or datetime.now(timezone.utc)).astimezone(timezone.utc)\n    age = now_utc - checked_at\n    if age < -MAX_FUTURE_SKEW:\n        raise ValueError("receipt check time is implausibly in the future")\n    if age > MAX_RECEIPT_AGE:\n        raise ValueError("admission receipt is older than 15 minutes; execution remains blocked")\n\n    return {\n        "verification": "manual_operator_attestation_no_provider_quota_api",\n        "account": DESIGNATED_KAGGLE_ACCOUNT,\n        "checked_at_utc": checked_at.isoformat().replace("+00:00", "Z"),\n        "notebook_sha256": expected_notebook_sha256,\n        "runner_sha256": actual_runner_sha256,\n        "intended_arm": intended_arm,\n        "free_t4_admission_confirmed_by_operator": True,\n        "paid_prompt_seen_by_operator": False,\n        "admission_provenance": provenance.strip(),\n        "evidence_sha256": receipt["evidence_sha256"],\n        "notebook_hash_verification": "operator_verified_local_pin_not_kernel_rehashed",\n        "runner_hash_verification": (\n            "rehash_executing_file" if runner_path is not None else "matched_pinned_hash_before_install"\n        ),\n        "provider_quota_api_queried": False,\n    }\n'
helper_path = Path("/kaggle/working/ditraj_admission.py")
helper_path.write_text(ADMISSION_MODULE_SOURCE, encoding="utf-8")
sys.path.insert(0, "/kaggle/working")
sys.modules.pop("ditraj_admission", None)
from ditraj_admission import validate_admission_receipt

PAIRED_RUNNER_SHA256 = "c00e1470c3e67da3573f61614f90e3ec9ab525135040ca3928a4bc3db4432984"
FEEDBACK_RUNNER_SHA256 = "8b00b8c413b6e8dcfd4b61ceaf206ec4a3f45351557c3b21aa52d06c75dac25f"
ADMISSION_RECEIPT_PATH = Path("/kaggle/working/ditraj_admission_receipt.json")
print("Copy the exact reviewed local notebook SHA-256 from the source registry; the kernel cannot re-hash the .ipynb file.")
ADMITTED_NOTEBOOK_SHA256 = input("Exact reviewed local notebook SHA-256: ").strip().lower()
setup_receipt = input("Paste fresh JSON receipt with intended_arm=setup_only: ").strip()
ADMISSION_RECEIPT_PATH.write_text(setup_receipt, encoding="utf-8")
setup_admission = validate_admission_receipt(
    ADMISSION_RECEIPT_PATH,
    intended_arm="setup_only",
    expected_notebook_sha256=ADMITTED_NOTEBOOK_SHA256,
    expected_runner_sha256=PAIRED_RUNNER_SHA256,
)
print(json.dumps(setup_admission, indent=2))

def collect_ditraj_receipt(intended_arm, expected_runner_sha256):
    raw = input(f"Paste a fresh (<15 min) no-paid receipt for {intended_arm}: ").strip()
    ADMISSION_RECEIPT_PATH.write_text(raw, encoding="utf-8")
    validated = validate_admission_receipt(
        ADMISSION_RECEIPT_PATH,
        intended_arm=intended_arm,
        expected_notebook_sha256=ADMITTED_NOTEBOOK_SHA256,
        expected_runner_sha256=expected_runner_sha256,
    )
    print(json.dumps(validated, indent=2))
    return str(ADMISSION_RECEIPT_PATH)

# No setup side effect runs unless the receipt above passes.
%cd /kaggle/working
!git clone https://github.com/xduzhangjiayu/DiTraj.git DiTraj
!git -C /kaggle/working/DiTraj checkout 6fa8e613c9e7c40ec2db456ab2ba34367a076886
!git clone --branch v0.33.1 --depth 1 https://github.com/huggingface/diffusers.git diffusers-v0.33.1
!pip install -r /kaggle/working/DiTraj/requirements.txt
!pip install -e /kaggle/working/diffusers-v0.33.1


In [ ]:
from importlib.metadata import distribution
import subprocess
repo = "/kaggle/working/DiTraj"
print("DiTraj commit:", subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"], text=True).strip())
print("Diffusers:", distribution("diffusers").version)
assert distribution("diffusers").version == "0.33.1"


In [ ]:
# Obtain a fresh receipt before touching torch.cuda or reading the device.
receipt_for_preflight = collect_ditraj_receipt("runtime_preflight", PAIRED_RUNNER_SHA256)
import platform
import torch
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; stop before downloading model weights.")
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU 0:", gpu_name)
print("Capability:", capability)
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Visible GPU count:", torch.cuda.device_count())
if "T4" not in gpu_name:
    raise RuntimeError(f"Expected a T4 on GPU 0, found {gpu_name!r}; stop before downloading model weights.")
if capability != (7, 5):
    raise RuntimeError(f"Expected T4 compute capability (7, 5); found {capability!r}; stop before downloading model weights.")
print("T4 capability preflight: PASS (7, 5)")


In [ ]:
# Stage exactly the reviewed local runner/module bytes into the pinned checkout.
from pathlib import Path
import hashlib

RUNNER_PINS = {
    "run_ditraj_paired_path_pilot.py": "c00e1470c3e67da3573f61614f90e3ec9ab525135040ca3928a4bc3db4432984",
    "run_ditraj_feedback_pilot.py": "8b00b8c413b6e8dcfd4b61ceaf206ec4a3f45351557c3b21aa52d06c75dac25f",
}
STAGED_SOURCES = {
    "run_ditraj_paired_path_pilot.py": '#!/usr/bin/env python3\n"""Direction-balanced six-case 17-frame DiTraj pilot for Wan2.1-T2V-1.3B.\n\nRun from the pinned DiTraj checkout after installing its pinned Diffusers\nenvironment and replacing Diffusers\' Wan transformer with DiTraj\'s copy:\n\n    python run_ditraj_paired_path_pilot.py --num-steps 1 \\\n        --output-dir /kaggle/working/ditraj_paired_path_smoke \\\n        --admission-receipt /path/to/fresh-receipt.json \\\n        --notebook-sha256 <reviewed-local-notebook-sha256>\n\nThis is an adapted small pilot, not an exact reproduction of the paper. It\nkeeps DiTraj\'s official pipeline/attention code and source behavior, uses the\none direction-neutral prompt, mirrored paths, a 17-frame shape, a 30-step sampler,\nand common initial latent noise across Base, SG-only, and full DiTraj arms. Research-only:\nthe upstream DiTraj repository states CC BY-NC 4.0.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport platform\nimport re\nimport secrets\nimport shutil\nimport statistics\nimport subprocess\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\ntry:  # works as both `research.*` and a staged flat runner\n    from .ditraj_admission import validate_admission_receipt\nexcept ImportError:\n    from ditraj_admission import validate_admission_receipt\n\n\nMODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"\nMODEL_REVISION = "0fad780a534b6463e45facd96134c9f345acfa5b"\nDITRAJ_COMMIT = "6fa8e613c9e7c40ec2db456ab2ba34367a076886"\nWIDTH, HEIGHT, NUM_FRAMES, NUM_STEPS = 832, 480, 17, 30\nSEED, GUIDANCE_SCALE, FLOW_SHIFT = 2026, 5.0, 3.0\nPROMPT = (\n    "A single glossy bright-red ball on a matte white tabletop, locked-off camera, "\n    "soft even daylight. The ball moves smoothly and stays fully visible in frame."\n)\nFG_PROMPT = "A single glossy bright-red ball moves smoothly and stays fully visible in frame."\nBG_PROMPT = "A locked-off camera, matte white tabletop, soft even daylight, simple clean background."\nNEGATIVE_PROMPT = "text, watermark, blurry image, JPEG artifacts"\nPATHS_YXYX = {\n    "right": [[0, 0.50, 0.76, 0.43, 0.57], [16, 0.50, 0.76, 0.63, 0.77]],\n    "left": [[0, 0.50, 0.76, 0.43, 0.57], [16, 0.50, 0.76, 0.23, 0.37]],\n}\nARM_CONFIGS = {\n    "base": {"mask_step": 0, "fixRope_step": 0},\n    "sg_only": {"mask_step": 18, "fixRope_step": 0},\n    "ditraj": {"mask_step": 18, "fixRope_step": 3},\n}\n\n\ndef sha256_tensor(tensor: torch.Tensor) -> str:\n    raw = tensor.detach().to("cpu").contiguous().numpy().tobytes()\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef package_version(name: str) -> str | None:\n    try:\n        return importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError:\n        return None\n\n\ndef normalized_target_boxes(n_frames: int, anchors: list[list[float]]) -> list[list[float]]:\n    """Interpolate anchors and return normalized [x1,y1,x2,y2] boxes."""\n    out: list[list[float]] = []\n    for frame in range(n_frames):\n        if frame <= anchors[0][0]:\n            row = anchors[0]\n            y1, y2, x1, x2 = row[1:]\n        elif frame >= anchors[-1][0]:\n            row = anchors[-1]\n            y1, y2, x1, x2 = row[1:]\n        else:\n            right = next(i for i, row in enumerate(anchors[1:], start=1) if frame <= row[0])\n            a, b = anchors[right - 1], anchors[right]\n            alpha = (frame - a[0]) / (b[0] - a[0])\n            y1, y2, x1, x2 = [a[i] + alpha * (b[i] - a[i]) for i in range(1, 5)]\n        out.append([x1, y1, x2, y2])\n    return out\n\n\ndef evaluate_red_proxy(frames_dir: Path, anchors: list[list[float]]) -> dict[str, Any]:\n    """Secondary red-pixel screen; blinded manual labels remain the primary score."""\n    paths = sorted(frames_dir.glob("frame_*.png"))\n    if len(paths) != NUM_FRAMES:\n        raise RuntimeError(f"Expected {NUM_FRAMES} saved frames, found {len(paths)} in {frames_dir}")\n    target_boxes = normalized_target_boxes(len(paths), anchors)\n    centers: list[tuple[float, float] | None] = []\n    areas: list[float] = []\n    widths: list[float] = []\n    heights: list[float] = []\n    valid_errors: list[float] = []\n    penalized_errors: list[float] = []\n    ious: list[float] = []\n    width_px = height_px = 0\n\n    for index, path in enumerate(paths):\n        rgb = np.asarray(Image.open(path).convert("RGB"))\n        h, w = rgb.shape[:2]\n        height_px, width_px = h, w\n        mask = (rgb[:, :, 0] > 150) & (rgb[:, :, 1] < 125) & (rgb[:, :, 2] < 125)\n        ys, xs = np.nonzero(mask)\n        if len(xs) < 300:\n            centers.append(None)\n            penalized_errors.append(1.0)\n            continue\n\n        x1, x2 = float(xs.min()), float(xs.max() + 1)\n        y1, y2 = float(ys.min()), float(ys.max() + 1)\n        cx, cy = float(xs.mean()), float(ys.mean())\n        centers.append((cx, cy))\n        areas.append(float(len(xs)))\n        widths.append(x2 - x1)\n        heights.append(y2 - y1)\n\n        tx1n, ty1n, tx2n, ty2n = target_boxes[index]\n        tx1, ty1, tx2, ty2 = tx1n * w, ty1n * h, tx2n * w, ty2n * h\n        tcx, tcy = (tx1 + tx2) / 2, (ty1 + ty2) / 2\n        error = math.hypot(cx - tcx, cy - tcy)\n        diag = math.hypot(w, h)\n        valid_errors.append(error / diag)\n        penalized_errors.append(error / diag)\n        ix1, iy1 = max(x1, tx1), max(y1, ty1)\n        ix2, iy2 = min(x2, tx2), min(y2, ty2)\n        intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)\n        union = (x2 - x1) * (y2 - y1) + (tx2 - tx1) * (ty2 - ty1) - intersection\n        ious.append(intersection / union if union else 0.0)\n\n    adjacent = [(a, b) for a, b in zip(centers, centers[1:]) if a is not None and b is not None]\n    target_dx = ((target_boxes[-1][0] + target_boxes[-1][2]) - (target_boxes[0][0] + target_boxes[0][2])) / 2\n    expected_sign = 1 if target_dx > 0 else -1\n    aligned = [expected_sign * (b[0] - a[0]) > 0 for a, b in adjacent]\n\n    def cv(values: list[float]) -> float | None:\n        return statistics.pstdev(values) / statistics.mean(values) if values and statistics.mean(values) else None\n\n    return {\n        "evaluator": "red_rgb_threshold_proxy_secondary_only",\n        "threshold": {"r_gt": 150, "g_lt": 125, "b_lt": 125, "minimum_pixels": 300},\n        "frame_count": len(paths),\n        "frame_size_wh": [width_px, height_px],\n        "valid_frame_coverage": len([center for center in centers if center is not None]) / len(paths),\n        "mean_center_error_frame_diagonals_valid_only": statistics.mean(valid_errors) if valid_errors else None,\n        "mean_center_error_frame_diagonals_missing_penalty_1": statistics.mean(penalized_errors),\n        "mean_bbox_iou_valid_only": statistics.mean(ious) if ious else None,\n        "red_mask_area_cv": cv(areas),\n        "bbox_width_cv": cv(widths),\n        "bbox_height_cv": cv(heights),\n        "direction_aligned_adjacent_visible_step_fraction": sum(aligned) / len(aligned) if aligned else None,\n        "adjacent_visible_step_count": len(aligned),\n        "limitations": [\n            "All red pixels are grouped; this does not track a semantic object instance.",\n            "Image-plane scoring does not verify depth, 3D pose, contact, or physical rolling.",\n            "Specular highlights can split or remove threshold pixels; blinded manual labels are primary.",\n        ],\n    }\n\ndef save_frames(frames: np.ndarray, out_dir: Path) -> list[str]:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    files = []\n    for index, frame in enumerate(frames):\n        pixels = (np.clip(frame, 0.0, 1.0) * 255).round().astype(np.uint8)\n        path = out_dir / f"frame_{index:03d}.png"\n        Image.fromarray(pixels, mode="RGB").save(path, optimize=True)\n        files.append(path.name)\n    return files\n\n\ndef save_contact_sheet(frame_dirs: dict[str, Path], target: Path) -> None:\n    arms = list(frame_dirs)\n    frames = [[Image.open(frame_dirs[arm] / f"frame_{i:03d}.png").convert("RGB") for i in range(NUM_FRAMES)] for arm in arms]\n    thumb_w, thumb_h = 208, 120\n    gap, label_h = 6, 22\n    columns = 5\n    rows = math.ceil(NUM_FRAMES / columns)\n    sheet = Image.new(\n        "RGB",\n        (columns * thumb_w + (columns + 1) * gap, len(arms) * (rows * (thumb_h + label_h) + gap) + gap),\n        "white",\n    )\n    draw = ImageDraw.Draw(sheet)\n    for arm_i, arm in enumerate(arms):\n        row_y = gap + arm_i * (rows * (thumb_h + label_h) + gap)\n        for frame_i, image in enumerate(frames[arm_i]):\n            col, row = frame_i % columns, frame_i // columns\n            x = gap + col * (thumb_w + gap)\n            y = row_y + row * (thumb_h + label_h)\n            image.thumbnail((thumb_w, thumb_h))\n            sheet.paste(image, (x, y + label_h))\n            draw.text((x + 2, y + 3), f"{arm} · frame {frame_i:02d}", fill="black")\n    sheet.save(target)\n\n\ndef save_annotation_template(output_dir: Path, frame_files: dict[str, list[str]]) -> None:\n    package_dir = output_dir / "blinded_annotation_package"\n    package_dir.mkdir(parents=True, exist_ok=True)\n    import csv\n\n    blind_ids = [f"clip_{chr(ord(\'A\') + index)}" for index in range(len(frame_files))]\n    secrets.SystemRandom().shuffle(blind_ids)\n    blind_map = dict(zip(frame_files, blind_ids, strict=True))\n    for arm, blind_id in blind_map.items():\n        dest = package_dir / blind_id\n        dest.mkdir(exist_ok=True)\n        for name in frame_files[arm]:\n            shutil.copy2(output_dir / arm / name, dest / name)\n    path = package_dir / "annotation_template.csv"\n    with path.open("w", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow([\n            "blind_clip_id", "frame_index", "image_file", "object_visible", "visible_reason_or_occlusion",\n            "instance_count", "same_identity_as_frame_0", "center_x_px", "center_y_px",\n            "bbox_x1_px", "bbox_y1_px", "bbox_x2_px", "bbox_y2_px",\n            "annotation_confidence_1_to_5", "blur_or_shape_note",\n        ])\n        for run_key, blind_id in blind_map.items():\n            for index, name in enumerate(frame_files[run_key]):\n                writer.writerow([blind_id, index, f"{blind_id}/{name}", "", "", "", "", "", "", "", "", "", "", "", ""])\n    key_path = output_dir.parent / "ditraj_private_unblind_key.json"\n    key_path.write_text(json.dumps({blind: arm for arm, blind in blind_map.items()}, indent=2) + "\\n")\n    # Preliminary local-only directory: do not archive/export until both the\n    # registered per-frame manifest and separate background measure exist.\n\n\ndef patch_upstream_source(repo_root: Path) -> dict[str, str]:\n    """Apply behavior-preserving runtime fixes to the pinned source checkout.\n\n    The CPU bound lookup returns the same inclusive corners for each filled\n    rectangular mask but avoids thousands of Python-level CUDA scalar reads.\n    The two zero-cutoff checks skip masks that cannot be consumed by that arm.\n    The published inclusive-corner/exclusive-slice behavior is intentionally\n    preserved for this first source-faithful comparison.\n    """\n    if not (repo_root / "module/pipe.py").is_file() or not (repo_root / "module/transformer_wan.py").is_file():\n        raise FileNotFoundError(f"Run from the pinned DiTraj checkout; expected module/ under {repo_root}")\n    head = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True).strip()\n    if head != DITRAJ_COMMIT:\n        raise RuntimeError(f"Expected DiTraj commit {DITRAJ_COMMIT}, found {head}")\n    diffusers_version = importlib.metadata.version("diffusers")\n    if diffusers_version != "0.33.1":\n        raise RuntimeError(f"Expected pinned Diffusers 0.33.1, found {diffusers_version}")\n\n    finder_pattern = re.compile(\n        r"(?m)^(?P<indent> *)def find_corners\\(matrix\\):\\n.*?^(?P=indent)    return top_left, bottom_right\\n",\n        re.DOTALL,\n    )\n\n    def vectorized_finder(match: re.Match[str]) -> str:\n        indent = match.group("indent")\n        return (\n            f"{indent}def find_corners(matrix):\\n"\n            f"{indent}    coords = torch.nonzero(matrix.detach().to(device=\'cpu\') == 1, as_tuple=False)\\n"\n            f"{indent}    if coords.numel() == 0:\\n"\n            f"{indent}        return None, None\\n"\n            f"{indent}    lo = coords.min(dim=0).values\\n"\n            f"{indent}    hi = coords.max(dim=0).values\\n"\n            f"{indent}    return (int(lo[0]), int(lo[1])), (int(hi[0]), int(hi[1]))\\n"\n        )\n\n    patched_hashes: dict[str, str] = {}\n    for relative in ("module/pipe.py", "module/transformer_wan.py"):\n        path = repo_root / relative\n        source = path.read_text()\n        if "coords = torch.nonzero(matrix.detach().to(device=\'cpu\') == 1" not in source:\n            source, count = finder_pattern.subn(vectorized_finder, source)\n            if count != 1:\n                raise RuntimeError(f"Expected one find_corners block in {relative}, replaced {count}")\n        if relative == "module/pipe.py":\n            source = source.replace(\n                "cross_attention_mask = self.create_cross_attention_mask(bbox_mask, attention_kwargs[\'encoder_attention_mask\'].flatten())",\n                "cross_attention_mask = (self.create_cross_attention_mask(bbox_mask, attention_kwargs[\'encoder_attention_mask\'].flatten()) if mask_step > 0 else None)",\n                1,\n            )\n            if "if mask_step > 0 else None" not in source:\n                raise RuntimeError("Could not apply the zero-mask-step cross-attention optimization")\n            source = source.replace(\n                "fixrope_self_attention_mask = self.create_fixrope_self_attention_mask(latents.shape[-3:], bbox_mask)",\n                "fixrope_self_attention_mask = (self.create_fixrope_self_attention_mask(latents.shape[-3:], bbox_mask) if fixRope_step > 0 else None)",\n                1,\n            )\n            if "if fixRope_step > 0 else None" not in source:\n                raise RuntimeError("Could not apply the zero-RoPE-step mask optimization")\n        path.write_text(source)\n        patched_hashes[relative] = hashlib.sha256(path.read_bytes()).hexdigest()\n    # The published setup replaces Diffusers\' Wan transformer with DiTraj\'s\n    # implementation. Copy the CPU-bound lookup optimization into that active\n    # module as well; otherwise the pipeline file would be fast but the actual\n    # loaded model would retain the per-pixel CUDA synchronization loop.\n    transformer_target = Path(\n        importlib.metadata.distribution("diffusers").locate_file(\n            "diffusers/models/transformers/transformer_wan.py"\n        )\n    )\n    shutil.copy2(repo_root / "module/transformer_wan.py", transformer_target)\n    patched_hashes["diffusers/models/transformers/transformer_wan.py"] = hashlib.sha256(\n        transformer_target.read_bytes()\n    ).hexdigest()\n    return patched_hashes\n\n\ndef runtime_metadata() -> dict[str, Any]:\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU is required; refusing to download model weights on CPU.")\n    name = torch.cuda.get_device_name(0)\n    capability = torch.cuda.get_device_capability(0)\n    total_memory = torch.cuda.get_device_properties(0).total_memory\n    if "T4" not in name:\n        raise RuntimeError(f"This ticket is scoped to a T4; observed device is {name!r}.")\n    return {\n        "python": platform.python_version(),\n        "torch": torch.__version__,\n        "torch_cuda": torch.version.cuda,\n        "diffusers": package_version("diffusers"),\n        "transformers": package_version("transformers"),\n        "accelerate": package_version("accelerate"),\n        "cuda_available": True,\n        "gpu": name,\n        "gpu_capability": list(capability),\n        "gpu_total_memory_bytes": total_memory,\n        "active_gpu_index": 0,\n        "torch_dtype": "float16",\n    }\n\n\ndef run(\n    output_dir: Path,\n    num_steps: int = NUM_STEPS,\n    *,\n    admission_receipt: Path,\n    notebook_sha256: str,\n) -> None:\n    admission_metadata = validate_admission_receipt(\n        admission_receipt,\n        intended_arm=f"paired_path_steps_{num_steps}",\n        expected_notebook_sha256=notebook_sha256,\n        runner_path=Path(__file__).resolve(),\n    )\n\n    # No model/CUDA-facing imports occur before the fresh receipt is checked.\n    global np, torch, Image, ImageDraw\n    import numpy as np\n    import torch\n    from PIL import Image, ImageDraw\n\n    repo_root = Path.cwd()\n    source_hashes = patch_upstream_source(repo_root)\n\n    # Delayed imports allow the notebook to install and patch Diffusers first.\n    from diffusers import AutoencoderKLWan\n    from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler\n    from module.attention_processor import MyWanAttnProcessor2_0\n    from module.pipe import myWanPipeline\n    from utils import plan_path\n\n    runtime = runtime_metadata()\n    run_started_at = datetime.now(timezone.utc).isoformat()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    device = torch.device("cuda:0")\n    print(json.dumps(runtime, indent=2), flush=True)\n\n    # Load the released DiTraj pipeline and use T4-safe FP16 transformer weights.\n    vae = AutoencoderKLWan.from_pretrained(\n        MODEL_ID,\n        revision=MODEL_REVISION,\n        subfolder="vae",\n        torch_dtype=torch.float32,\n        low_cpu_mem_usage=True,\n    )\n    pipe = myWanPipeline.from_pretrained(\n        MODEL_ID,\n        revision=MODEL_REVISION,\n        vae=vae,\n        torch_dtype=torch.float16,\n        low_cpu_mem_usage=True,\n    )\n    processors = {name: MyWanAttnProcessor2_0() for name in pipe.transformer.attn_processors.keys()}\n    pipe.transformer.set_attn_processor(processors)\n    pipe.scheduler = UniPCMultistepScheduler(\n        prediction_type="flow_prediction",\n        use_flow_sigmas=True,\n        num_train_timesteps=1000,\n        flow_shift=FLOW_SHIFT,\n    )\n    pipe.model_cpu_offload_seq = "text_encoder->transformer->vae"\n    pipe.enable_model_cpu_offload(gpu_id=0)\n\n    # Wan\'s temporal compression maps 17 frames to five path masks. Build one\n    # mask per mirrored target below while holding prompt and initial noise fixed.\n    latent_frames = NUM_FRAMES // 4 + 1\n    box_h, box_w = HEIGHT // 16, WIDTH // 16\n\n    # One initial noise tensor is explicitly shared across all three arms.\n    init_generator = torch.Generator(device=device).manual_seed(SEED)\n    common_latents = pipe.prepare_latents(\n        batch_size=1,\n        num_channels_latents=int(pipe.transformer.config.in_channels),\n        height=HEIGHT,\n        width=WIDTH,\n        num_frames=NUM_FRAMES,\n        dtype=torch.float32,\n        device=device,\n        generator=init_generator,\n        latents=None,\n    )\n    common_noise_sha = sha256_tensor(common_latents)\n    torch.save({"latents": common_latents.detach().cpu(), "seed": SEED}, output_dir / "common_initial_noise.pt")\n\n    arms: dict[str, dict[str, Any]] = {}\n    frame_dirs: dict[str, Path] = {}\n    frame_files: dict[str, list[str]] = {}\n    for path_name, anchors in PATHS_YXYX.items():\n        bbox_mask = torch.zeros((latent_frames, 1, box_h, box_w), device=device)\n        pixel_path = plan_path(anchors, video_length=NUM_FRAMES)[::4]\n        if len(pixel_path) != latent_frames:\n            raise RuntimeError(f"Expected {latent_frames} latent boxes for {path_name}, got {len(pixel_path)}")\n        for index, (y1, y2, x1, x2) in enumerate(pixel_path):\n            y1i, y2i = int(y1 * box_h), int(y2 * box_h)\n            x1i, x2i = int(x1 * box_w), int(x2 * box_w)\n            bbox_mask[index, :, y1i:y2i, x1i:x2i] = 1\n        if not bool(torch.all(bbox_mask.flatten(1).sum(dim=1) > 0)):\n            raise RuntimeError(f"At least one latent-frame {path_name} box is empty")\n\n        for arm_name, control in ARM_CONFIGS.items():\n            run_key = f"{path_name}_{arm_name}"\n            # Recreate scheduler state; every arm and direction receives the\n            # same prompt, model revision, seed, and explicitly shared noise.\n            pipe.scheduler = UniPCMultistepScheduler(\n                prediction_type="flow_prediction",\n                use_flow_sigmas=True,\n                num_train_timesteps=1000,\n                flow_shift=FLOW_SHIFT,\n            )\n            torch.cuda.empty_cache()\n            torch.cuda.reset_peak_memory_stats(0)\n            arm_generator = torch.Generator(device=device).manual_seed(SEED)\n            start = time.perf_counter()\n            with torch.inference_mode():\n                output = pipe(\n                    prompt=PROMPT,\n                    negative_prompt=NEGATIVE_PROMPT,\n                    height=HEIGHT,\n                    width=WIDTH,\n                    num_frames=NUM_FRAMES,\n                    num_inference_steps=num_steps,\n                    guidance_scale=GUIDANCE_SCALE,\n                    generator=arm_generator,\n                    latents=common_latents.clone(),\n                    output_type="np",\n                    attention_kwargs={\n                        "bbox_mask": bbox_mask,\n                        "encoder_attention_mask": torch.tensor(\n                            [False] * 512 + [True] * 512, device=device\n                        ),\n                        "bg_prompt": BG_PROMPT,\n                        "fg_prompt": FG_PROMPT,\n                        "fixRope_step": control["fixRope_step"],\n                        "mask_step": control["mask_step"],\n                    },\n                )\n            elapsed = time.perf_counter() - start\n            frames = np.asarray(output.frames[0], dtype=np.float32)\n            if frames.shape != (NUM_FRAMES, HEIGHT, WIDTH, 3):\n                raise RuntimeError(f"Unexpected {run_key} output frame shape: {frames.shape}")\n            arm_dir = output_dir / run_key\n            names = save_frames(frames, arm_dir)\n            frame_dirs[run_key] = arm_dir\n            frame_files[run_key] = names\n            metrics = evaluate_red_proxy(arm_dir, anchors)\n            arms[run_key] = {\n                "path_condition": path_name,\n                "settings": control,\n                "initial_noise_sha256": common_noise_sha,\n                "generation_seconds": elapsed,\n                "peak_vram_allocated_bytes": torch.cuda.max_memory_allocated(0),\n                "peak_vram_reserved_bytes": torch.cuda.max_memory_reserved(0),\n                "frame_paths": [f"{run_key}/{name}" for name in names],\n                "secondary_proxy_metrics": metrics,\n            }\n            del output, frames\n            torch.cuda.empty_cache()\n            print(f"Completed {run_key}: {elapsed:.1f}s, {metrics[\'valid_frame_coverage\']:.0%} proxy coverage", flush=True)\n\n    save_contact_sheet(frame_dirs, output_dir / "contact_sheet.png")\n    save_annotation_template(output_dir, frame_files)\n    report = {\n        "experiment_id": "EXP-DITRAJ-WAN13B-PAIRED-PATH-PILOT-002",\n        "status": "completed_single_seed_screen" if num_steps == NUM_STEPS else "runtime_smoke_only",\n        "run_started_at_utc": run_started_at,\n        "run_completed_at_utc": datetime.now(timezone.utc).isoformat(),\n        "adaptation_note": "17-frame/30-step DiTraj adaptation; not an exact reproduction of the 81-frame paper setup.",\n        "model": {"repo_id": MODEL_ID, "revision": MODEL_REVISION},\n        "ditraj": {"commit": DITRAJ_COMMIT, "source_repo": "https://github.com/xduzhangjiayu/DiTraj"},\n        "local_source_adaptations": {\n            "patched_file_sha256": source_hashes,\n            "runner_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n            "changes": [\n                "Vectorized the filled-box corner lookup on CPU while preserving upstream inclusive bounds and exclusive slicing.",\n                "Skipped cross-attention and repeated-token masks only when their corresponding cutoff is zero.",\n            ],\n        },\n        "runtime": runtime,\n        "account_quota": {\n            **admission_metadata,\n            "quota_hours_before": "not queried; this runner has no Kaggle quota API",\n            "quota_hours_after": "not queried; this runner has no Kaggle quota API",\n        },\n        "generation": {\n            "prompt": PROMPT,\n            "foreground_prompt": FG_PROMPT,\n            "background_prompt": BG_PROMPT,\n            "direction_words_in_prompt": False,\n            "negative_prompt": NEGATIVE_PROMPT,\n            "seed": SEED,\n            "width": WIDTH,\n            "height": HEIGHT,\n            "num_frames": NUM_FRAMES,\n            "num_inference_steps": num_steps,\n            "guidance_scale": GUIDANCE_SCALE,\n            "scheduler": "UniPCMultistepScheduler(flow_prediction, flow_shift=3.0)",\n            "same_initial_noise_sha256_across_arms": common_noise_sha,\n            "target_paths_yxyx": PATHS_YXYX,\n        },\n        "arms": arms,\n        "primary_evaluation": {\n            "method": "blinded_manual per-frame visibility, identity/count, center, bbox, confidence, and shape labels",\n            "primary_metrics": ["center_ADE_normalized_by_frame_diagonal", "center_FDE_normalized_by_frame_diagonal"],\n            "missing_detection_center_error_penalty": 1.0,\n            "template_directory": "blinded_annotation_package/",\n            "status": "preliminary_not_scoreable",\n            "scoreable": False,\n            "exportable": False,\n            "missing_requirements": [\n                "registered per-frame SHA-256 and dimension manifest",\n                "separate background-stability measure",\n            ],\n            "unblinding_key": "../ditraj_private_unblind_key.json (keep separate until labels are complete)",\n            "rule": "Do not export or score this preliminary directory; a later registered packet must hide path direction and arm names and treat missing detections as failures.",\n        },\n        "decision_gate": {\n            "runtime": "all arms produce 17 frames with identical initial noise and recorded runtime/memory",\n            "pilot_signal": "full DiTraj lowers blinded center ADE versus both controls when averaged over mirrored paths, with >=90% visibility and the correct signed endpoint response in both directions",\n            "guardrails": "no identity loss/freeze/disappearance; no material apparent-size, background, or human-rated quality regression",\n            "caveat": "single seed is a screening result only; repeat >=5 paired seeds before claiming improvement",\n        },\n        "licenses": {\n            "ditraj": "CC BY-NC 4.0 per upstream README; research-only pending any separate commercial-use review",\n            "wan": "See the pinned model card and its terms",\n        },\n    }\n    (output_dir / "pilot_report.json").write_text(json.dumps(report, indent=2) + "\\n")\n    print(f"Artifacts saved to {output_dir}", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/ditraj_paired_path_pilot"))\n    parser.add_argument("--num-steps", type=int, default=NUM_STEPS)\n    parser.add_argument("--admission-receipt", type=Path, required=True)\n    parser.add_argument("--notebook-sha256", required=True)\n    args = parser.parse_args()\n    if not 1 <= args.num_steps <= NUM_STEPS:\n        parser.error(f"--num-steps must be between 1 and {NUM_STEPS}")\n    run(\n        args.output_dir.resolve(),\n        num_steps=args.num_steps,\n        admission_receipt=args.admission_receipt.resolve(),\n        notebook_sha256=args.notebook_sha256,\n    )\n\n\nif __name__ == "__main__":\n    main()\n',
    "run_ditraj_feedback_pilot.py": '#!/usr/bin/env python3\n"""Small one-path probe of a step-4 x0-hat feedback gate in DiTraj/Wan.\n\nThis keeps the paired open-loop runner unchanged. It runs a full DiTraj\nbaseline, a decode-matched zero-update control, and one feedback arm from the\nsame noise. The default eight-step screen is for checking the intervention\npath, mask change, runtime, and VRAM only; it is not an efficacy benchmark.\n\nRun from the exact pinned DiTraj checkout after the paired-path setup has\ninstalled DiTraj\'s Diffusers transformer copy and required dependencies:\n\n    python run_ditraj_feedback_pilot.py \\\n        --output-dir /kaggle/working/feedback_smoke \\\n        --admission-receipt /path/to/fresh-receipt.json \\\n        --notebook-sha256 <reviewed-local-notebook-sha256>\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\ntry:  # flat imports on Kaggle; package imports in the local research workspace\n    from . import run_ditraj_paired_path_pilot as baseline\n    from .ditraj_admission import validate_admission_receipt\nexcept ImportError:\n    import run_ditraj_paired_path_pilot as baseline\n    from ditraj_admission import validate_admission_receipt\n\n\nARM_NAMES = ("ditraj_baseline", "decode_matched_noop", "x0_hat_feedback")\n\n\ndef make_path_mask(path_name: str, *, plan_path: Any, device: torch.device) -> torch.Tensor:\n    height, width = baseline.HEIGHT, baseline.WIDTH\n    anchors = baseline.PATHS_YXYX[path_name]\n    latent_frames = baseline.NUM_FRAMES // 4 + 1\n    box_h, box_w = height // 16, width // 16\n    pixel_path = plan_path(anchors, video_length=baseline.NUM_FRAMES)[::4]\n    if len(pixel_path) != latent_frames:\n        raise RuntimeError(f"Expected {latent_frames} latent boxes for {path_name}, got {len(pixel_path)}")\n    bbox_mask = torch.zeros((latent_frames, 1, box_h, box_w), device=device)\n    for index, (y1, y2, x1, x2) in enumerate(pixel_path):\n        y1i, y2i = int(y1 * box_h), int(y2 * box_h)\n        x1i, x2i = int(x1 * box_w), int(x2 * box_w)\n        bbox_mask[index, :, y1i:y2i, x1i:x2i] = 1\n    if not bool(torch.all(bbox_mask.flatten(1).sum(dim=1) > 0)):\n        raise RuntimeError(f"At least one latent-frame {path_name} box is empty")\n    return bbox_mask\n\n\ndef run(\n    output_dir: Path,\n    *,\n    path_name: str = "right",\n    num_steps: int = 8,\n    admission_receipt: Path,\n    notebook_sha256: str,\n) -> None:\n    admission_metadata = validate_admission_receipt(\n        admission_receipt,\n        intended_arm=f"feedback_path_{path_name}_steps_{num_steps}",\n        expected_notebook_sha256=notebook_sha256,\n        runner_path=Path(__file__).resolve(),\n    )\n\n    global np, torch\n    import numpy as np\n    import torch\n    try:\n        from .ditraj_feedback_adapter import FEEDBACK_STEP, RedPathFeedback, patch_pinned_pipeline\n    except ImportError:\n        from ditraj_feedback_adapter import FEEDBACK_STEP, RedPathFeedback, patch_pinned_pipeline\n\n    if path_name not in baseline.PATHS_YXYX:\n        raise ValueError(f"path_name must be one of {tuple(baseline.PATHS_YXYX)}")\n    if not FEEDBACK_STEP < num_steps <= baseline.NUM_STEPS:\n        raise ValueError(f"num_steps must be {FEEDBACK_STEP + 1}..{baseline.NUM_STEPS}")\n    repo_root = Path.cwd()\n    source_hashes = baseline.patch_upstream_source(repo_root)\n    feedback_pipe_hash = patch_pinned_pipeline(repo_root / "module/pipe.py")\n    source_hashes["module/pipe.py_feedback_patch"] = feedback_pipe_hash\n\n    # Imports happen only after the notebook\'s setup and source patching.\n    from diffusers import AutoencoderKLWan\n    from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler\n    from module.attention_processor import MyWanAttnProcessor2_0\n    from module.pipe import myWanPipeline\n    from utils import plan_path\n\n    runtime = baseline.runtime_metadata()\n    started = datetime.now(timezone.utc).isoformat()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    device = torch.device("cuda:0")\n    print(json.dumps(runtime, indent=2), flush=True)\n\n    vae = AutoencoderKLWan.from_pretrained(\n        baseline.MODEL_ID,\n        revision=baseline.MODEL_REVISION,\n        subfolder="vae",\n        torch_dtype=torch.float32,\n        low_cpu_mem_usage=True,\n    )\n    pipe = myWanPipeline.from_pretrained(\n        baseline.MODEL_ID,\n        revision=baseline.MODEL_REVISION,\n        vae=vae,\n        torch_dtype=torch.float16,\n        low_cpu_mem_usage=True,\n    )\n    processors = {name: MyWanAttnProcessor2_0() for name in pipe.transformer.attn_processors.keys()}\n    pipe.transformer.set_attn_processor(processors)\n    pipe.scheduler = UniPCMultistepScheduler(\n        prediction_type="flow_prediction",\n        use_flow_sigmas=True,\n        num_train_timesteps=1000,\n        flow_shift=baseline.FLOW_SHIFT,\n    )\n    pipe.model_cpu_offload_seq = "text_encoder->transformer->vae"\n    pipe.enable_model_cpu_offload(gpu_id=0)\n\n    target_mask = make_path_mask(path_name, plan_path=plan_path, device=device)\n    initial_generator = torch.Generator(device=device).manual_seed(baseline.SEED)\n    common_latents = pipe.prepare_latents(\n        batch_size=1,\n        num_channels_latents=int(pipe.transformer.config.in_channels),\n        height=baseline.HEIGHT,\n        width=baseline.WIDTH,\n        num_frames=baseline.NUM_FRAMES,\n        dtype=torch.float32,\n        device=device,\n        generator=initial_generator,\n        latents=None,\n    )\n    noise_sha256 = baseline.sha256_tensor(common_latents)\n    torch.save({"latents": common_latents.detach().cpu(), "seed": baseline.SEED}, output_dir / "common_initial_noise.pt")\n\n    arms: dict[str, dict[str, Any]] = {}\n    frame_dirs: dict[str, Path] = {}\n    frame_files: dict[str, list[str]] = {}\n    for arm_name in ARM_NAMES:\n        pipe.scheduler = UniPCMultistepScheduler(\n            prediction_type="flow_prediction",\n            use_flow_sigmas=True,\n            num_train_timesteps=1000,\n            flow_shift=baseline.FLOW_SHIFT,\n        )\n        feedback = None\n        if arm_name == "decode_matched_noop":\n            feedback = RedPathFeedback(initial_bbox_mask=target_mask, gain=0.0)\n        elif arm_name == "x0_hat_feedback":\n            feedback = RedPathFeedback(initial_bbox_mask=target_mask, gain=0.5)\n\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats(0)\n        generator = torch.Generator(device=device).manual_seed(baseline.SEED)\n        start = time.perf_counter()\n        try:\n            with torch.inference_mode():\n                kwargs = {\n                    "prompt": baseline.PROMPT,\n                    "negative_prompt": baseline.NEGATIVE_PROMPT,\n                    "height": baseline.HEIGHT,\n                    "width": baseline.WIDTH,\n                    "num_frames": baseline.NUM_FRAMES,\n                    "num_inference_steps": num_steps,\n                    "guidance_scale": baseline.GUIDANCE_SCALE,\n                    "generator": generator,\n                    "latents": common_latents.clone(),\n                    "output_type": "np",\n                    "attention_kwargs": {\n                        "bbox_mask": target_mask.clone(),\n                        "encoder_attention_mask": torch.tensor([False] * 512 + [True] * 512, device=device),\n                        "bg_prompt": baseline.BG_PROMPT,\n                        "fg_prompt": baseline.FG_PROMPT,\n                        "fixRope_step": 3,\n                        "mask_step": 18,\n                    },\n                }\n                if feedback is not None:\n                    kwargs["attention_kwargs"]["latentgate_feedback"] = feedback\n                output = pipe(**kwargs)\n        finally:\n            if feedback is not None:\n                feedback.close()\n        elapsed = time.perf_counter() - start\n        if arm_name != "ditraj_baseline" and feedback.report is None:\n            raise RuntimeError(f"{arm_name} completed without a step-{FEEDBACK_STEP} observer report")\n        frames = np.asarray(output.frames[0], dtype=np.float32)\n        if frames.shape != (baseline.NUM_FRAMES, baseline.HEIGHT, baseline.WIDTH, 3):\n            raise RuntimeError(f"Unexpected {arm_name} frame shape: {frames.shape}")\n        frame_dir = output_dir / arm_name\n        names = baseline.save_frames(frames, frame_dir)\n        frames_sha256 = hashlib.sha256(np.ascontiguousarray(frames).tobytes()).hexdigest()\n        frame_dirs[arm_name] = frame_dir\n        frame_files[arm_name] = names\n        metrics = baseline.evaluate_red_proxy(frame_dir, baseline.PATHS_YXYX[path_name])\n        arms[arm_name] = {\n            "path_condition": path_name,\n            "same_initial_noise_sha256": noise_sha256,\n            "decoded_frame_array_sha256": frames_sha256,\n            "generation_seconds": elapsed,\n            "peak_vram_allocated_bytes": torch.cuda.max_memory_allocated(0),\n            "peak_vram_reserved_bytes": torch.cuda.max_memory_reserved(0),\n            "frame_paths": [f"{arm_name}/{name}" for name in names],\n            "secondary_proxy_metrics": metrics,\n            "feedback_report": feedback.report if feedback is not None else None,\n        }\n        del output, frames\n        torch.cuda.empty_cache()\n        print(\n            f"Completed {arm_name}: {elapsed:.1f}s; "\n            f"{metrics[\'valid_frame_coverage\']:.0%} red-proxy coverage",\n            flush=True,\n        )\n\n    noop_matches_baseline = (\n        arms["ditraj_baseline"]["decoded_frame_array_sha256"]\n        == arms["decode_matched_noop"]["decoded_frame_array_sha256"]\n    )\n    feedback_mask_changed = (\n        int(\n            arms["x0_hat_feedback"]["feedback_report"]["rasterization"]["changed_token_total"]\n        )\n        > 0\n    )\n    baseline.save_contact_sheet(frame_dirs, output_dir / "contact_sheet.png")\n    baseline.save_annotation_template(output_dir, frame_files)\n    report = {\n        "experiment_id": "EXP-DITRAJ-WAN13B-X0-BOX-FEEDBACK-MECHANISM-SMOKE-001",\n        "status": "mechanism_smoke_only_not_an_efficacy_test",\n        "run_started_at_utc": started,\n        "run_completed_at_utc": datetime.now(timezone.utc).isoformat(),\n        "model": {"repo_id": baseline.MODEL_ID, "revision": baseline.MODEL_REVISION},\n        "ditraj": {"commit": baseline.DITRAJ_COMMIT, "source_repo": "https://github.com/xduzhangjiayu/DiTraj"},\n        "local_source_sha256": source_hashes,\n        "runner_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        "admission": admission_metadata,\n        "account_quota": {\n            "quota_hours_before": "not queried; no Kaggle quota API is available to this runner",\n            "quota_hours_after": "not queried; no Kaggle quota API is available to this runner",\n        },\n        "annotation_packet": {\n            "status": "preliminary_not_scoreable",\n            "scoreable": False,\n            "exportable": False,\n            "missing_requirements": [\n                "registered per-frame SHA-256 and dimension manifest",\n                "separate background-stability measure",\n            ],\n        },\n        "runtime": runtime,\n        "generation": {\n            "prompt": baseline.PROMPT,\n            "negative_prompt": baseline.NEGATIVE_PROMPT,\n            "direction_words_in_prompt": False,\n            "path_condition": path_name,\n            "seed": baseline.SEED,\n            "width": baseline.WIDTH,\n            "height": baseline.HEIGHT,\n            "num_frames": baseline.NUM_FRAMES,\n            "num_inference_steps": num_steps,\n            "feedback_step_zero_based": FEEDBACK_STEP,\n            "guidance_scale": baseline.GUIDANCE_SCALE,\n            "scheduler": "UniPCMultistepScheduler(flow_prediction, flow_shift=3.0)",\n            "target_paths_yxyx": baseline.PATHS_YXYX[path_name],\n            "initial_noise_sha256_shared_across_arms": noise_sha256,\n            "arms": list(ARM_NAMES),\n        },\n        "decision_boundary": [\n            "This screen checks whether exact x0_hat capture, VAE decoding, observer output, bounded box update, and cached cross-attention-mask refresh work together.",\n            "The decode-matched no-op control estimates effects from the extra VAE decode and observer schedule.",\n            "A single path and seed with at most a few post-update steps cannot establish an efficacy claim.",\n            "The threshold observer is red-object-specific and is not independent ground truth.",\n            "Review local-only preliminary frames; do not export or score them until the registered frame manifest and separate background-stability measure exist.",\n        ],\n        "arms": arms,\n        "control_checks": {\n            "decode_matched_noop_exactly_matches_baseline_frames": noop_matches_baseline,\n            "feedback_changed_any_rasterized_attention_tokens": feedback_mask_changed,\n            "all_mechanism_gates_passed": noop_matches_baseline and feedback_mask_changed,\n            "failures": [\n                name\n                for name, passed in (\n                    ("decode_matched_noop_exactly_matches_baseline_frames", noop_matches_baseline),\n                    ("feedback_changed_any_rasterized_attention_tokens", feedback_mask_changed),\n                )\n                if not passed\n            ],\n            "note": "A changed mask verifies actuation plumbing only; it does not prove that generated object motion improves.",\n        },\n    }\n    if not report["control_checks"]["all_mechanism_gates_passed"]:\n        report["status"] = "mechanism_smoke_gate_failed_not_an_efficacy_test"\n    (output_dir / "feedback_smoke_report.json").write_text(json.dumps(report, indent=2) + "\\n")\n    print(f"Artifacts saved to {output_dir}", flush=True)\n    if not report["control_checks"]["all_mechanism_gates_passed"]:\n        failed = ", ".join(report["control_checks"]["failures"])\n        raise RuntimeError(f"Feedback mechanism smoke gate failed: {failed}")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/ditraj_feedback_smoke"))\n    parser.add_argument("--path", choices=("right", "left"), default="right")\n    parser.add_argument("--num-steps", type=int, default=8)\n    parser.add_argument("--admission-receipt", type=Path, required=True)\n    parser.add_argument("--notebook-sha256", required=True)\n    args = parser.parse_args()\n    run(\n        args.output_dir.resolve(),\n        path_name=args.path,\n        num_steps=args.num_steps,\n        admission_receipt=args.admission_receipt.resolve(),\n        notebook_sha256=args.notebook_sha256,\n    )\n\n\nif __name__ == "__main__":\n    main()\n',
    "ditraj_admission.py": '"""Fail-closed validation for a manually reviewed, no-paid DiTraj admission receipt.\n\nThis module does not query Kaggle billing, quota, or account state. It validates\nthat an operator\'s fresh attestation is complete and matches the intended arm,\nthe locally pinned notebook hash, and the exact runner bytes before any runner\nimports model, CUDA, or upstream-patching code.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nfrom typing import Any\n\n\nRECEIPT_SCHEMA = "latentgate.ditraj_free_admission.v1"\nDESIGNATED_KAGGLE_ACCOUNT = "bvyashwantkumar"\nMAX_RECEIPT_AGE = timedelta(minutes=15)\nMAX_FUTURE_SKEW = timedelta(minutes=2)\nSHA256_RE = re.compile(r"^[0-9a-f]{64}$")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _parse_utc(value: Any) -> datetime:\n    if not isinstance(value, str) or not value.strip():\n        raise ValueError("receipt checked_at_utc must be an ISO-8601 UTC timestamp")\n    normalized = value.strip()\n    if normalized.endswith("Z"):\n        normalized = normalized[:-1] + "+00:00"\n    try:\n        parsed = datetime.fromisoformat(normalized)\n    except ValueError as exc:\n        raise ValueError("receipt checked_at_utc must be an ISO-8601 UTC timestamp") from exc\n    if parsed.tzinfo is None or parsed.utcoffset() != timedelta(0):\n        raise ValueError("receipt checked_at_utc must include UTC timezone information")\n    return parsed.astimezone(timezone.utc)\n\n\ndef validate_admission_receipt(\n    receipt_path: Path,\n    *,\n    intended_arm: str,\n    expected_notebook_sha256: str,\n    runner_path: Path | None = None,\n    expected_runner_sha256: str | None = None,\n    now: datetime | None = None,\n) -> dict[str, Any]:\n    """Validate a manual receipt; returns a JSON-safe, explicit provenance record.\n\n    The notebook file is not available to a Kaggle kernel for independent\n    hashing. Therefore `expected_notebook_sha256` must be transcribed from the\n    reviewed local source pin; the receipt records that operator check. During\n    pre-install notebook setup, the runner hash is compared with its pinned\n    local hash. At CLI invocation, the executing runner file is re-hashed.\n    """\n    if not intended_arm or not intended_arm.strip():\n        raise ValueError("an explicit intended arm is required")\n    if not SHA256_RE.fullmatch(expected_notebook_sha256):\n        raise ValueError("expected notebook SHA-256 must be 64 lowercase hex characters")\n    if not receipt_path.is_file():\n        raise ValueError(f"no admission receipt at {receipt_path}; execution remains blocked")\n    if expected_runner_sha256 is not None and not SHA256_RE.fullmatch(expected_runner_sha256):\n        raise ValueError("expected runner SHA-256 must be 64 lowercase hex characters")\n    if runner_path is None and expected_runner_sha256 is None:\n        raise ValueError("an expected runner hash or runner file is required")\n    if runner_path is not None and not runner_path.is_file():\n        raise ValueError(f"runner source is missing at {runner_path}; execution remains blocked")\n\n    try:\n        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))\n    except (OSError, json.JSONDecodeError) as exc:\n        raise ValueError("admission receipt is unreadable or invalid JSON") from exc\n    if not isinstance(receipt, dict):\n        raise ValueError("admission receipt must be a JSON object")\n\n    required = {\n        "schema",\n        "account",\n        "checked_at_utc",\n        "notebook_sha256",\n        "runner_sha256",\n        "intended_arm",\n        "free_t4_admission_confirmed",\n        "paid_prompt_seen",\n        "admission_provenance",\n        "evidence_sha256",\n    }\n    missing = sorted(required - receipt.keys())\n    if missing:\n        raise ValueError(f"admission receipt is missing required fields: {\', \'.join(missing)}")\n    if receipt["schema"] != RECEIPT_SCHEMA:\n        raise ValueError("admission receipt schema is not supported")\n    if receipt["account"] != DESIGNATED_KAGGLE_ACCOUNT:\n        raise ValueError("receipt account does not match the designated Kaggle account")\n    if receipt["intended_arm"] != intended_arm:\n        raise ValueError("receipt intended_arm does not match this run")\n    if receipt["free_t4_admission_confirmed"] is not True:\n        raise ValueError("fresh free T4 admission is not attested; execution remains blocked")\n    if receipt["paid_prompt_seen"] is not False:\n        raise ValueError("a paid/upgrade prompt was seen or not explicitly ruled out")\n    if receipt["notebook_sha256"] != expected_notebook_sha256:\n        raise ValueError("receipt notebook SHA-256 does not match the reviewed local notebook pin")\n    if not isinstance(receipt["runner_sha256"], str) or not SHA256_RE.fullmatch(receipt["runner_sha256"]):\n        raise ValueError("receipt runner SHA-256 must be 64 lowercase hex characters")\n\n    actual_runner_sha256 = sha256_file(runner_path) if runner_path is not None else expected_runner_sha256\n    if receipt["runner_sha256"] != actual_runner_sha256:\n        raise ValueError("receipt runner SHA-256 does not match the executing runner bytes")\n    if not isinstance(receipt["evidence_sha256"], str) or not SHA256_RE.fullmatch(receipt["evidence_sha256"]):\n        raise ValueError("receipt evidence_sha256 must identify the reviewed admission evidence")\n    provenance = receipt["admission_provenance"]\n    if not isinstance(provenance, str) or len(provenance.strip()) < 12:\n        raise ValueError("receipt admission_provenance must describe the manual UI evidence")\n\n    checked_at = _parse_utc(receipt["checked_at_utc"])\n    now_utc = (now or datetime.now(timezone.utc)).astimezone(timezone.utc)\n    age = now_utc - checked_at\n    if age < -MAX_FUTURE_SKEW:\n        raise ValueError("receipt check time is implausibly in the future")\n    if age > MAX_RECEIPT_AGE:\n        raise ValueError("admission receipt is older than 15 minutes; execution remains blocked")\n\n    return {\n        "verification": "manual_operator_attestation_no_provider_quota_api",\n        "account": DESIGNATED_KAGGLE_ACCOUNT,\n        "checked_at_utc": checked_at.isoformat().replace("+00:00", "Z"),\n        "notebook_sha256": expected_notebook_sha256,\n        "runner_sha256": actual_runner_sha256,\n        "intended_arm": intended_arm,\n        "free_t4_admission_confirmed_by_operator": True,\n        "paid_prompt_seen_by_operator": False,\n        "admission_provenance": provenance.strip(),\n        "evidence_sha256": receipt["evidence_sha256"],\n        "notebook_hash_verification": "operator_verified_local_pin_not_kernel_rehashed",\n        "runner_hash_verification": (\n            "rehash_executing_file" if runner_path is not None else "matched_pinned_hash_before_install"\n        ),\n        "provider_quota_api_queried": False,\n    }\n',
    "ditraj_feedback_adapter.py": '"""Opt-in x0-hat feedback adapter for the pinned DiTraj Wan pipeline.\n\nThis module is deliberately separate from the published/open-loop runner. It\ncaptures UniPC\'s exact converted clean prediction at one selected step, decodes\nit with Wan\'s VAE, observes a controlled red object, and returns a bounded\nforeground-box update for the remaining spatial-guidance steps.\n\nThe red observer and stability score are suitable only for the red-ball pilot.\nThe score is a threshold-agreement heuristic, not a calibrated probability.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport math\nimport statistics\nfrom pathlib import Path\nfrom typing import Any, Sequence\n\nimport numpy as np\n\ntry:  # supports both `research.*` imports and flat notebook staging\n    from .box_residual_feedback import FeedbackConfig, correct_path_boxes\nexcept ImportError:\n    from box_residual_feedback import FeedbackConfig, correct_path_boxes\n\n\nPINNED_DITRAJ_COMMIT = "6fa8e613c9e7c40ec2db456ab2ba34367a076886"\nFEEDBACK_STEP = 4\nOBSERVATION_FRAMES = (0, 4, 8, 12, 16)\nTHRESHOLDS = tuple((r, gb) for r in (130, 150, 170) for gb in (100, 125, 150))\nMIN_RED_PIXELS = 300\nREFERENCE_FRAME_AREA = 832 * 480\n\n\ndef minimum_red_pixels(height: int, width: int) -> int:\n    """Scale the canonical 300-pixel floor across fixture/output resolutions."""\n    return max(8, int(round(MIN_RED_PIXELS * height * width / REFERENCE_FRAME_AREA)))\n\n\ndef _replace_once(source: str, old: str, new: str, label: str) -> str:\n    count = source.count(old)\n    if count != 1:\n        raise RuntimeError(f"Expected exactly one {label} source marker, found {count}")\n    return source.replace(old, new, 1)\n\n\ndef patch_pinned_pipeline(pipe_path: Path) -> str:\n    """Add fail-closed feedback hooks to the already pinned DiTraj pipe source.\n\n    The caller must independently verify the git commit and apply the baseline\n    runner\'s documented performance patches first. This function also works on\n    the unoptimized pinned source for local patch-shape checks.\n    """\n    source = pipe_path.read_text()\n    if "latentgate_feedback.after_scheduler_step" in source:\n        required = (\n            "latentgate_feedback.before_scheduler_step",\n            "latentgate_feedback.after_scheduler_step",\n            "segment_mask = attention_kwargs[\'encoder_attention_mask\'].flatten()",\n            "updated_bbox_mask is not None and not torch.equal(updated_bbox_mask, bbox_mask)",\n        )\n        if not all(marker in source for marker in required):\n            raise RuntimeError("Partial feedback patch detected in module/pipe.py")\n        compile(source, str(pipe_path), "exec")\n        return hashlib.sha256(pipe_path.read_bytes()).hexdigest()\n\n    source = _replace_once(\n        source,\n        "        mask_step = attention_kwargs[\'mask_step\']\\n",\n        "        mask_step = attention_kwargs[\'mask_step\']\\n"\n        "        latentgate_feedback = attention_kwargs.pop(\'latentgate_feedback\', None)\\n"\n        "        segment_mask = attention_kwargs[\'encoder_attention_mask\'].flatten()\\n"\n        "        if latentgate_feedback is not None and not (\\n"\n        "            hasattr(latentgate_feedback, \'before_scheduler_step\')\\n"\n        "            and hasattr(latentgate_feedback, \'after_scheduler_step\')\\n"\n        "        ):\\n"\n        "            raise TypeError(\\"latentgate_feedback must implement the pinned feedback adapter interface\\")\\n",\n        "feedback-controller setup",\n    )\n    source = _replace_once(\n        source,\n        "self.create_cross_attention_mask(bbox_mask, attention_kwargs[\'encoder_attention_mask\'].flatten())",\n        "self.create_cross_attention_mask(bbox_mask, segment_mask)",\n        "cross-attention segment mask",\n    )\n    source = _replace_once(\n        source,\n        "                latents = self.scheduler.step(noise_pred, t, latents, return_dict=False)[0]\\n",\n        "                if latentgate_feedback is not None:\\n"\n        "                    latentgate_feedback.before_scheduler_step(self.scheduler, i)\\n"\n        "                latents = self.scheduler.step(noise_pred, t, latents, return_dict=False)[0]\\n"\n        "                if latentgate_feedback is not None:\\n"\n        "                    updated_bbox_mask = latentgate_feedback.after_scheduler_step(\\n"\n        "                        self, i, t, bbox_mask\\n"\n        "                    )\\n"\n        "                    if updated_bbox_mask is not None and not torch.equal(updated_bbox_mask, bbox_mask):\\n"\n        "                        bbox_mask = updated_bbox_mask\\n"\n        "                        cross_attention_mask = self.create_cross_attention_mask(bbox_mask, segment_mask)\\n",\n        "scheduler feedback hook",\n    )\n    compile(source, str(pipe_path), "exec")\n    pipe_path.write_text(source)\n    return hashlib.sha256(pipe_path.read_bytes()).hexdigest()\n\n\ndef boxes_from_mask(mask: Any) -> list[list[float]]:\n    """Recover normalized xyxy rectangle boxes from DiTraj\'s [T,1,H,W] mask."""\n    array = mask.detach().to(device="cpu").numpy()\n    if array.ndim != 4 or array.shape[1] != 1:\n        raise ValueError(f"Expected bbox mask [T,1,H,W], got {tuple(array.shape)}")\n    _, _, height, width = array.shape\n    boxes: list[list[float]] = []\n    for index, frame in enumerate(array[:, 0] > 0.5):\n        ys, xs = np.nonzero(frame)\n        if not len(xs):\n            raise ValueError(f"bbox mask frame {index} is empty")\n        # The pilot uses solid rectangles. Refuse malformed masks instead of\n        # expanding a disconnected foreground into an unintended box.\n        crop = frame[ys.min() : ys.max() + 1, xs.min() : xs.max() + 1]\n        if not crop.all() or int(frame.sum()) != int(crop.size):\n            raise ValueError(f"bbox mask frame {index} is not one filled rectangle")\n        boxes.append([\n            float(xs.min()) / width,\n            float(ys.min()) / height,\n            float(xs.max() + 1) / width,\n            float(ys.max() + 1) / height,\n        ])\n    return boxes\n\n\ndef rasterize_like_ditraj(boxes_xyxy: Sequence[Sequence[float]], *, height: int, width: int) -> np.ndarray:\n    """Rasterize using DiTraj\'s int/floor lower and upper endpoint semantics."""\n    boxes = np.asarray(boxes_xyxy, dtype=np.float64)\n    if boxes.ndim != 2 or boxes.shape[1] != 4 or not np.isfinite(boxes).all():\n        raise ValueError("boxes_xyxy must be finite [T,4]")\n    if np.any(boxes < 0.0) or np.any(boxes > 1.0):\n        raise ValueError("box coordinates must lie in [0,1]")\n    mask = np.zeros((len(boxes), 1, height, width), dtype=np.float32)\n    for frame, (x1, y1, x2, y2) in enumerate(boxes):\n        # Keep the baseline\'s int() behavior (floor for nonnegative inputs).\n        # The tiny epsilon only neutralizes float roundoff when a box came\n        # back from an integer token boundary (e.g. 17/30*30 == 16.999999...).\n        left = min(width - 1, max(0, int(np.floor(x1 * width + 1e-9))))\n        top = min(height - 1, max(0, int(np.floor(y1 * height + 1e-9))))\n        right = min(width, max(left + 1, int(np.floor(x2 * width + 1e-9))))\n        bottom = min(height, max(top + 1, int(np.floor(y2 * height + 1e-9))))\n        mask[frame, 0, top:bottom, left:right] = 1.0\n    return mask\n\n\ndef mask_change_report(before: Any, after: Any) -> list[dict[str, float | int]]:\n    """Measure the actual discrete intervention on DiTraj\'s latent token grid."""\n    old = before.detach().to(device="cpu").numpy()[:, 0] > 0.5\n    new = after.detach().to(device="cpu").numpy()[:, 0] > 0.5\n    if old.shape != new.shape:\n        raise ValueError("before/after masks must have the same shape")\n    records: list[dict[str, float | int]] = []\n    for frame, (a, b) in enumerate(zip(old, new, strict=True)):\n        ay, ax = np.nonzero(a)\n        by, bx = np.nonzero(b)\n        intersection = int(np.logical_and(a, b).sum())\n        union = int(np.logical_or(a, b).sum())\n        records.append({\n            "latent_frame": frame,\n            "changed_token_count": int(np.logical_xor(a, b).sum()),\n            "target_occupied_tokens": int(a.sum()),\n            "updated_occupied_tokens": int(b.sum()),\n            "mask_iou": intersection / union if union else 0.0,\n            "center_shift_tokens_xy": [\n                float(bx.mean() - ax.mean()), float(by.mean() - ay.mean())\n            ],\n        })\n    return records\n\n\ndef observe_red_frame(rgb_float: np.ndarray) -> dict[str, Any]:\n    """Return a full-frame nine-threshold red centroid and agreement heuristic."""\n    if rgb_float.ndim != 3 or rgb_float.shape[-1] != 3:\n        raise ValueError(f"Expected RGB image [H,W,3], got {rgb_float.shape}")\n    height, width = rgb_float.shape[:2]\n    rgb = (np.clip(rgb_float, 0.0, 1.0) * 255.0).round().astype(np.uint8)\n    minimum_pixels = minimum_red_pixels(height, width)\n    centers: list[tuple[float, float]] = []\n    areas: list[int] = []\n    for red_min, gb_max in THRESHOLDS:\n        mask = (rgb[:, :, 0] > red_min) & (rgb[:, :, 1] < gb_max) & (rgb[:, :, 2] < gb_max)\n        ys, xs = np.nonzero(mask)\n        if len(xs) >= minimum_pixels:\n            centers.append((float(xs.mean()), float(ys.mean())))\n            areas.append(len(xs))\n    if not centers:\n        return {\n            "scoreable": False,\n            "valid_threshold_count": 0,\n            "threshold_count": len(THRESHOLDS),\n            "minimum_pixels_per_threshold": minimum_pixels,\n            "stability_heuristic": 0.0,\n            "center_xy_normalized": None,\n            "center_spread_frame_diagonals": None,\n        }\n    cx = statistics.median(point[0] for point in centers)\n    cy = statistics.median(point[1] for point in centers)\n    diagonal = math.hypot(width, height)\n    spread = statistics.median(math.hypot(x - cx, y - cy) for x, y in centers) / diagonal\n    coverage = len(centers) / len(THRESHOLDS)\n    stability = coverage * max(0.0, 1.0 - spread / 0.02)\n    mean_area = statistics.mean(areas)\n    return {\n        "scoreable": True,\n        "valid_threshold_count": len(centers),\n        "threshold_count": len(THRESHOLDS),\n        "minimum_pixels_per_threshold": minimum_pixels,\n        "stability_heuristic": stability,\n        "center_xy_normalized": [cx / width, cy / height],\n        "center_spread_frame_diagonals": spread,\n        "area_cv_across_thresholds": statistics.pstdev(areas) / mean_area if mean_area else None,\n    }\n\n\ndef decode_wan_x0_hat(pipe: Any, normalized_latents: Any) -> np.ndarray:\n    """Inverse-normalize and decode Wan x0_hat into [frames,H,W,RGB] in [0,1]."""\n    import torch\n\n    vae = pipe.vae\n    device = pipe._execution_device\n    vae_dtype = next(vae.parameters()).dtype\n    latents = normalized_latents.to(device=device, dtype=torch.float32)\n    channels = int(vae.config.z_dim)\n    mean = torch.tensor(vae.config.latents_mean, dtype=torch.float32, device=device).view(1, channels, 1, 1, 1)\n    std = torch.tensor(vae.config.latents_std, dtype=torch.float32, device=device).view(1, channels, 1, 1, 1)\n    raw_latents = (latents * std + mean).to(dtype=vae_dtype)\n    with torch.inference_mode():\n        decoded = vae.decode(raw_latents, return_dict=False)[0]\n    rgb = ((decoded.float().clamp(-1.0, 1.0) + 1.0) * 0.5)\n    return rgb.permute(0, 2, 3, 4, 1)[0].cpu().numpy()\n\n\nclass UniPCX0Capture:\n    """Capture the exact scheduler conversion for one UniPC denoising step."""\n\n    def __init__(self, scheduler: Any, capture_step: int):\n        if type(scheduler).__name__ != "UniPCMultistepScheduler":\n            raise RuntimeError(f"Expected UniPCMultistepScheduler, got {type(scheduler).__name__}")\n        config = dict(scheduler.config)\n        if config.get("prediction_type") != "flow_prediction" or config.get("predict_x0") is not True:\n            raise RuntimeError(\n                "Refusing feedback: scheduler must use flow_prediction with predict_x0=True; "\n                f"got prediction_type={config.get(\'prediction_type\')!r}, predict_x0={config.get(\'predict_x0\')!r}"\n            )\n        if not callable(getattr(scheduler, "convert_model_output", None)):\n            raise RuntimeError("Pinned UniPC scheduler has no convert_model_output method")\n        self.scheduler = scheduler\n        self.capture_step = int(capture_step)\n        self.active_step: int | None = None\n        self.captured: dict[str, Any] | None = None\n        self.original_convert = scheduler.convert_model_output\n        self._closed = False\n\n        def convert_with_capture(model_output: Any, *args: Any, **kwargs: Any):\n            converted = self.original_convert(model_output, *args, **kwargs)\n            step = self.active_step\n            if step == self.capture_step:\n                sigma_index = getattr(self.scheduler, "step_index", None)\n                if sigma_index is None or int(sigma_index) != step:\n                    raise RuntimeError(\n                        f"UniPC sigma index mismatch: requested step {step}, step_index={sigma_index!r}"\n                    )\n                sample = kwargs.get("sample")\n                if sample is None and args:\n                    sample = args[0]\n                if sample is None:\n                    raise RuntimeError("UniPC conversion did not expose the pre-step sample")\n                self.captured = {\n                    "step_zero_based": step,\n                    "timestep": self.active_timestep,\n                    "scheduler_step_index": int(sigma_index),\n                    "x0_hat": converted.detach().clone(),\n                    "shape": list(converted.shape),\n                    "dtype": str(converted.dtype),\n                    "scheduler_config": config,\n                    "formula": "exact Diffusers UniPC convert_model_output result",\n                }\n            return converted\n\n        self._wrapped_convert = convert_with_capture\n        scheduler.convert_model_output = convert_with_capture\n        self.active_timestep: float | None = None\n\n    def begin(self, step: int, timestep: Any) -> None:\n        if self.active_step is not None:\n            raise RuntimeError("UniPC capture was armed while another scheduler step was active")\n        if step == self.capture_step:\n            self.captured = None\n            self.active_step = step\n            self.active_timestep = float(timestep.item()) if hasattr(timestep, "item") else float(timestep)\n\n    def consume(self, step: int) -> dict[str, Any]:\n        if step != self.capture_step or self.active_step != step:\n            raise RuntimeError(f"UniPC capture consumed outside selected step {self.capture_step}")\n        self.active_step = None\n        self.active_timestep = None\n        if self.captured is None:\n            raise RuntimeError("Pinned UniPC did not call convert_model_output at the feedback step")\n        payload = self.captured\n        self.captured = None\n        return payload\n\n    def close(self) -> None:\n        if self._closed:\n            return\n        self.scheduler.convert_model_output = self.original_convert\n        self._closed = True\n\n\nclass RedPathFeedback:\n    """Single-read, bounded foreground-box feedback for a five-frame path."""\n\n    def __init__(\n        self,\n        *,\n        initial_bbox_mask: Any,\n        observe_step: int = FEEDBACK_STEP,\n        gain: float = 0.5,\n        minimum_stability: float = 0.65,\n    ):\n        self.observe_step = int(observe_step)\n        self.target_boxes = boxes_from_mask(initial_bbox_mask)\n        self.target_mask = initial_bbox_mask.detach().clone()\n        _, _, self.mask_h, self.mask_w = self.target_mask.shape\n        if len(self.target_boxes) != len(OBSERVATION_FRAMES):\n            raise ValueError(\n                f"Expected {len(OBSERVATION_FRAMES)} latent-time boxes, got {len(self.target_boxes)}"\n            )\n        # 0.65 is an in-sample development cutoff on two correlated Kubric\n        # red-ball paths. It is neither calibrated nor validated for gating;\n        # generated-preview observer checks must freeze a cutoff before holdout.\n        self.config = FeedbackConfig(\n            gain=gain,\n            minimum_stability=minimum_stability,\n            max_shift_xy=(1.0 / self.mask_w, 1.0 / self.mask_h),\n            locked_box_indices=(0,),\n        )\n        self.capture: UniPCX0Capture | None = None\n        self.report: dict[str, Any] | None = None\n\n    def before_scheduler_step(self, scheduler: Any, step: int) -> None:\n        if step == self.observe_step:\n            if self.capture is None:\n                self.capture = UniPCX0Capture(scheduler, self.observe_step)\n            self.capture.begin(step, scheduler.timesteps[step])\n\n    def after_scheduler_step(self, pipe: Any, step: int, timestep: Any, current_bbox_mask: Any):\n        if step != self.observe_step:\n            return None\n        if self.capture is None:\n            raise RuntimeError("Feedback observer reached without an installed UniPC capture")\n        payload = self.capture.consume(step)\n        try:\n            preview = decode_wan_x0_hat(pipe, payload["x0_hat"])\n        finally:\n            # The VAE decode no longer needs the full x0 tensor; release it\n            # before the subsequent transformer steps on a small GPU.\n            del payload["x0_hat"]\n        if preview.shape[0] != 17:\n            raise RuntimeError(f"Expected 17 decoded x0_hat frames, got {preview.shape[0]}")\n        all_frame_observations = [observe_red_frame(frame) for frame in preview]\n        selected = [all_frame_observations[index] for index in OBSERVATION_FRAMES]\n        raw_centers = [row["center_xy_normalized"] for row in selected]\n        stability = [row["stability_heuristic"] for row in selected]\n        target_centers = [\n            [(box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0]\n            for box in self.target_boxes\n        ]\n        anchor_is_trusted = (\n            raw_centers[0] is not None\n            and stability[0] >= self.config.minimum_stability\n        )\n        anchor_offset = None\n        calibrated_centers: list[list[float] | None]\n        if anchor_is_trusted:\n            # The first box is locked. Use its observed-vs-commanded offset to\n            # cancel static color-centroid bias, then steer relative trajectory\n            # error. If the start observation is unreliable, fail closed.\n            anchor_offset = [\n                float(raw_centers[0][0] - target_centers[0][0]),\n                float(raw_centers[0][1] - target_centers[0][1]),\n            ]\n            calibrated_centers = []\n            for center in raw_centers:\n                if center is None:\n                    calibrated_centers.append(None)\n                    continue\n                corrected = [center[0] - anchor_offset[0], center[1] - anchor_offset[1]]\n                calibrated_centers.append(\n                    corrected if all(0.0 <= coordinate <= 1.0 for coordinate in corrected) else None\n                )\n        else:\n            calibrated_centers = [None] * len(raw_centers)\n        update = correct_path_boxes(\n            self.target_boxes,\n            calibrated_centers,\n            stability,\n            config=self.config,\n        )\n        active_boxes = update["steered_boxes_xyxy"]\n        new_mask_array = rasterize_like_ditraj(active_boxes, height=self.mask_h, width=self.mask_w)\n        import torch\n\n        updated_mask = torch.as_tensor(\n            new_mask_array,\n            dtype=current_bbox_mask.dtype,\n            device=current_bbox_mask.device,\n        )\n        mask_report = mask_change_report(self.target_mask, updated_mask)\n        self.report = {\n            "observe_step_zero_based": int(step),\n            "timestep": float(timestep.item()) if hasattr(timestep, "item") else float(timestep),\n            "x0_hat_capture": {key: value for key, value in payload.items() if key != "x0_hat"},\n            "observer": {\n                "family": "full-frame red threshold ensemble",\n                "red_gt": [130, 150, 170],\n                "green_blue_lt": [100, 125, 150],\n                "minimum_pixels_per_threshold_at_832x480": MIN_RED_PIXELS,\n                "minimum_pixel_area_scales_with_frame_area": True,\n                "all_17_frames": all_frame_observations,\n                "controller_rgb_frames": list(OBSERVATION_FRAMES),\n                "confidence_semantics": "threshold agreement heuristic, not calibrated probability",\n            },\n            "controller": update,\n            "start_anchor_calibration": {\n                "method": "subtract the observed-minus-commanded start offset from all later centers; controls relative path error",\n                "start_observation_trusted": bool(anchor_is_trusted),\n                "minimum_stability": self.config.minimum_stability,\n                "observed_start_offset_xy_normalized": anchor_offset,\n                "raw_observed_centers_xy_normalized": raw_centers,\n                "calibrated_observed_centers_xy_normalized": calibrated_centers,\n                "fallback": "if the first RGB observation is missing or below stability threshold, all boxes remain unchanged",\n            },\n            "rasterization": {\n                "semantics": "DiTraj int() floor/floor endpoint slices",\n                "latent_grid_wh": [self.mask_w, self.mask_h],\n                "target_vs_updated_per_latent_frame": mask_report,\n                "changed_token_total": sum(int(row["changed_token_count"]) for row in mask_report),\n            },\n        }\n        del preview, all_frame_observations\n        return updated_mask\n\n    def close(self) -> None:\n        if self.capture is not None:\n            self.capture.close()\n',
    "box_residual_feedback.py": '"""Bounded object-state residual controller for DiTraj-style path boxes.\n\nThis is a pure, isolated policy component. It is not yet wired into the Wan or\nDiTraj denoising loop. Coordinates are normalized ``xyxy`` image boxes and\nobserved object centers; a confidence score is used only as a threshold gate,\nnot as a calibrated probability.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Sequence\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass FeedbackConfig:\n    gain: float = 0.5\n    minimum_stability: float = 0.75\n    # One-cell cap for the current 52x30 Wan latent guidance grid.\n    max_shift_xy: tuple[float, float] = (1.0 / 52.0, 1.0 / 30.0)\n    locked_box_indices: tuple[int, ...] = (0,)\n\n    def validate(self) -> None:\n        if not 0.0 <= self.gain <= 1.0:\n            raise ValueError("gain must be in [0,1]")\n        if not 0.0 <= self.minimum_stability <= 1.0:\n            raise ValueError("minimum_stability must be in [0,1]")\n        if len(self.max_shift_xy) != 2 or any(not 0.0 <= x <= 1.0 for x in self.max_shift_xy):\n            raise ValueError("max_shift_xy must contain normalized nonnegative x/y limits")\n\n\ndef correct_path_boxes(\n    target_boxes_xyxy: Sequence[Sequence[float]],\n    observed_centers_xy: Sequence[Sequence[float] | None],\n    observer_stability: Sequence[float],\n    *,\n    config: FeedbackConfig = FeedbackConfig(),\n) -> dict[str, object]:\n    """Move trusted target boxes toward the signed residual from current x0-hat.\n\n    For each unlocked frame, ``delta = clip(gain * (target_center - observed),\n    +/- max_shift_xy)`` if the observation is present and its stability score\n    passes the configured threshold. The box size is preserved and the shifted\n    box is clamped inside the frame. Untrusted/missing observations get zero\n    correction; the initial box is locked by default.\n\n    ``observer_stability`` should be a diagnostic score such as threshold\n    ensemble agreement. It is not interpreted as a probability.\n    """\n    config.validate()\n    boxes = np.asarray(target_boxes_xyxy, dtype=np.float64)\n    if boxes.ndim != 2 or boxes.shape[1] != 4 or boxes.shape[0] == 0:\n        raise ValueError("target_boxes_xyxy must have shape [T,4]")\n    if not np.isfinite(boxes).all() or np.any(boxes < 0) or np.any(boxes > 1):\n        raise ValueError("target boxes must be finite and normalized to [0,1]")\n    if np.any(boxes[:, 0] >= boxes[:, 2]) or np.any(boxes[:, 1] >= boxes[:, 3]):\n        raise ValueError("target boxes must have positive width and height")\n    if len(observed_centers_xy) != len(boxes) or len(observer_stability) != len(boxes):\n        raise ValueError("Need one observation and stability score per target box")\n\n    stability = np.asarray(observer_stability, dtype=np.float64)\n    if not np.isfinite(stability).all() or np.any(stability < 0) or np.any(stability > 1):\n        raise ValueError("observer_stability values must be finite and in [0,1]")\n\n    centers = np.column_stack(((boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2))\n    observed = np.full_like(centers, np.nan)\n    for index, center in enumerate(observed_centers_xy):\n        if center is None:\n            continue\n        row = np.asarray(center, dtype=np.float64)\n        if row.shape != (2,):\n            raise ValueError(f"observed center {index} must contain normalized x and y")\n        if np.isfinite(row).all() and np.all(row >= 0) and np.all(row <= 1):\n            observed[index] = row\n\n    trusted = np.isfinite(observed).all(axis=1) & (stability >= config.minimum_stability)\n    locked = np.zeros(len(boxes), dtype=bool)\n    for index in config.locked_box_indices:\n        if not 0 <= int(index) < len(boxes):\n            raise ValueError(f"locked box index {index} is out of range")\n        locked[int(index)] = True\n    trusted &= ~locked\n\n    residual = centers - observed\n    residual[~np.isfinite(residual)] = 0.0\n    limits = np.asarray(config.max_shift_xy, dtype=np.float64)\n    shift = np.clip(config.gain * residual, -limits, limits)\n    shift[~trusted] = 0.0\n\n    widths = boxes[:, 2] - boxes[:, 0]\n    heights = boxes[:, 3] - boxes[:, 1]\n    shifted_centers = centers + shift\n    half_w, half_h = widths / 2, heights / 2\n    shifted_centers[:, 0] = np.clip(shifted_centers[:, 0], half_w, 1.0 - half_w)\n    shifted_centers[:, 1] = np.clip(shifted_centers[:, 1], half_h, 1.0 - half_h)\n    shifted_boxes = np.column_stack((\n        shifted_centers[:, 0] - half_w,\n        shifted_centers[:, 1] - half_h,\n        shifted_centers[:, 0] + half_w,\n        shifted_centers[:, 1] + half_h,\n    ))\n\n    return {\n        "target_boxes_xyxy": boxes.tolist(),\n        "steered_boxes_xyxy": shifted_boxes.tolist(),\n        "observed_centers_xy": [None if not np.isfinite(row).all() else row.tolist() for row in observed],\n        "target_residual_xy": residual.tolist(),\n        "applied_shift_xy": shift.tolist(),\n        "trusted_update": trusted.tolist(),\n        "locked_update": locked.tolist(),\n        "confidence_semantics": "observer stability threshold only; not a calibrated probability",\n        "config": asdict(config),\n    }\n\n\ndef rasterize_path_boxes(\n    boxes_xyxy: Sequence[Sequence[float]], *, height: int, width: int\n) -> np.ndarray:\n    """Rasterize normalized xyxy boxes into a bool mask shaped [T,H,W]."""\n    if height < 1 or width < 1:\n        raise ValueError("height and width must be positive")\n    boxes = np.asarray(boxes_xyxy, dtype=np.float64)\n    if boxes.ndim != 2 or boxes.shape[1] != 4 or boxes.shape[0] == 0:\n        raise ValueError("boxes_xyxy must have shape [T,4]")\n    if not np.isfinite(boxes).all() or np.any(boxes < 0) or np.any(boxes > 1):\n        raise ValueError("boxes must be finite and normalized to [0,1]")\n    mask = np.zeros((len(boxes), height, width), dtype=bool)\n    for index, (x1, y1, x2, y2) in enumerate(boxes):\n        left = min(width - 1, max(0, int(np.floor(x1 * width))))\n        top = min(height - 1, max(0, int(np.floor(y1 * height))))\n        right = min(width, max(left + 1, int(np.ceil(x2 * width))))\n        bottom = min(height, max(top + 1, int(np.ceil(y2 * height))))\n        mask[index, top:bottom, left:right] = True\n    return mask\n\n\ndef summarize_rasterized_updates(\n    target_boxes_xyxy: Sequence[Sequence[float]],\n    steered_boxes_xyxy: Sequence[Sequence[float]],\n    *,\n    height: int,\n    width: int,\n) -> list[dict[str, float | int]]:\n    """Report the effective token-grid change after box rasterization."""\n    target = rasterize_path_boxes(target_boxes_xyxy, height=height, width=width)\n    steered = rasterize_path_boxes(steered_boxes_xyxy, height=height, width=width)\n    if target.shape != steered.shape:\n        raise ValueError("target and steered paths must have the same number of boxes")\n    summaries: list[dict[str, float | int]] = []\n    for target_mask, steered_mask in zip(target, steered):\n        target_y, target_x = np.nonzero(target_mask)\n        steered_y, steered_x = np.nonzero(steered_mask)\n        intersection = int(np.logical_and(target_mask, steered_mask).sum())\n        union = int(np.logical_or(target_mask, steered_mask).sum())\n        summaries.append({\n            "target_occupied_tokens": int(target_mask.sum()),\n            "steered_occupied_tokens": int(steered_mask.sum()),\n            "changed_token_count": int(np.logical_xor(target_mask, steered_mask).sum()),\n            "foreground_mask_iou": intersection / union if union else 0.0,\n            "center_shift_tokens_x": float(steered_x.mean() - target_x.mean()),\n            "center_shift_tokens_y": float(steered_y.mean() - target_y.mean()),\n        })\n    return summaries\n',
}
for filename, content in STAGED_SOURCES.items():
    target = Path("/kaggle/working/DiTraj") / filename
    if filename in RUNNER_PINS:
        actual = hashlib.sha256(content.encode("utf-8")).hexdigest()
        if actual != RUNNER_PINS[filename]:
            raise RuntimeError(f"Embedded runner pin mismatch for {filename}: {actual}")
    target.write_text(content, encoding="utf-8")
    print("Staged reviewed source:", filename, hashlib.sha256(content.encode("utf-8")).hexdigest())

observer_calibration_source = '{\n  "experiment_id": "EXP-DITRAJ-RED-OBSERVER-KUBRIC-CALIBRATION-001",\n  "status": "known_truth_observer_calibration_cpu_only",\n  "created_at_utc": "2026-10-01T22:44:17.969184+00:00",\n  "fixture": {\n    "id": "EXP-LATENTGATE-KUBRIC-GATE0-RENDER-001",\n    "source_commit": "61f2422c84bab75006df33c6989e0b483db3ccfe",\n    "directions": [\n      "left",\n      "right"\n    ],\n    "frames": 34,\n    "frame_size_wh": [\n      160,\n      96\n    ],\n    "truth": "Kubric binary segmentation masks; observer receives RGB only",\n    "latent_control_grid_wh_for_relative_error": [\n      52,\n      30\n    ],\n    "relative_error_units": "continuous displacement scaled to the 52x30 control grid; not a quantized token error"\n  },\n  "observer": {\n    "family": "full-frame nine-threshold red centroid ensemble",\n    "thresholds_red_gt": [\n      130,\n      150,\n      170\n    ],\n    "thresholds_green_blue_lt": [\n      100,\n      125,\n      150\n    ],\n    "minimum_area": "300 pixels at 832x480, scaled by image area with an 8-pixel floor",\n    "center": "coordinate-wise median of scoreable threshold-specific centroids",\n    "stability": "threshold coverage times clipped 1 - median center spread / 0.02 frame diagonals; heuristic, not calibrated probability; 0.65 is an in-sample development cutoff, not a held-out pass gate"\n  },\n  "aggregate": {\n    "left": {\n      "frames": 17,\n      "scoreable_coverage": 1.0,\n      "mean_valid_center_error_frame_diagonals": 0.00926639912673367,\n      "p95_valid_center_error_frame_diagonals": 0.009782237393790468,\n      "maximum_valid_center_error_frame_diagonals": 0.009891498377518798,\n      "mean_missing_penalized_center_error_frame_diagonals": 0.00926639912673367,\n      "mean_valid_center_error_px": 1.729021681485457,\n      "p95_valid_center_error_px": 1.8252721813487707,\n      "first_frame_absolute_center_error_mean_px_missing_penalized": 1.7613287685906618,\n      "first_frame_absolute_center_error_p95_px_missing_penalized": 1.7613287685906618,\n      "stability_cutoff": 0.65,\n      "trusted_coverage": 0.9411764705882353,\n      "mean_trusted_center_error_frame_diagonals": 0.009235866437913376,\n      "mean_stability_heuristic": 0.7145905000255517,\n      "mean_valid_threshold_votes_out_of_9": 9,\n      "anchor_relative_trajectory_error_mean_abs_control_grid_units_xy": [\n        0.030769519169837847,\n        0.01916967743422975\n      ],\n      "anchor_relative_trajectory_error_p95_abs_control_grid_units_xy": [\n        0.06898327464788591,\n        0.04079356238273889\n      ],\n      "anchor_relative_trajectory_error_mean_l2_control_grid_units": 0.03886050371315209,\n      "controller_keyframe_count": 5,\n      "controller_keyframe_anchor_relative_mean_abs_control_grid_units_xy": [\n        0.02626538356351733,\n        0.01619446697977156\n      ],\n      "controller_keyframe_anchor_relative_p95_abs_control_grid_units_xy": [\n        0.05301274271844707,\n        0.04596650415312475\n      ],\n      "controller_keyframe_anchor_relative_mean_l2_control_grid_units": 0.03270587634183273\n    },\n    "right": {\n      "frames": 17,\n      "scoreable_coverage": 1.0,\n      "mean_valid_center_error_frame_diagonals": 0.009274088843827372,\n      "p95_valid_center_error_frame_diagonals": 0.010097902204096662,\n      "maximum_valid_center_error_frame_diagonals": 0.010124745399616372,\n      "mean_missing_penalized_center_error_frame_diagonals": 0.009274088843827372,\n      "mean_valid_center_error_px": 1.730456509340124,\n      "p95_valid_center_error_px": 1.8841722237100789,\n      "first_frame_absolute_center_error_mean_px_missing_penalized": 1.7613287685906618,\n      "first_frame_absolute_center_error_p95_px_missing_penalized": 1.7613287685906618,\n      "stability_cutoff": 0.65,\n      "trusted_coverage": 1.0,\n      "mean_trusted_center_error_frame_diagonals": 0.009274088843827372,\n      "mean_stability_heuristic": 0.7036123410258236,\n      "mean_valid_threshold_votes_out_of_9": 9,\n      "anchor_relative_trajectory_error_mean_abs_control_grid_units_xy": [\n        0.024748046953856937,\n        0.021252633826272287\n      ],\n      "anchor_relative_trajectory_error_p95_abs_control_grid_units_xy": [\n        0.08140302440640526,\n        0.05010574463899298\n      ],\n      "anchor_relative_trajectory_error_mean_l2_control_grid_units": 0.03702441036784015,\n      "controller_keyframe_count": 5,\n      "controller_keyframe_anchor_relative_mean_abs_control_grid_units_xy": [\n        0.003439900617780367,\n        0.014752436638218902\n      ],\n      "controller_keyframe_anchor_relative_p95_abs_control_grid_units_xy": [\n        0.0061090686274497315,\n        0.03759896347836033\n      ],\n      "controller_keyframe_anchor_relative_mean_l2_control_grid_units": 0.015964507397475815\n    },\n    "all": {\n      "frames": 34,\n      "scoreable_coverage": 1.0,\n      "mean_valid_center_error_frame_diagonals": 0.009270243985280521,\n      "p95_valid_center_error_frame_diagonals": 0.0100067550117426,\n      "maximum_valid_center_error_frame_diagonals": 0.010124745399616372,\n      "mean_missing_penalized_center_error_frame_diagonals": 0.009270243985280521,\n      "mean_valid_center_error_px": 1.7297390954127907,\n      "p95_valid_center_error_px": 1.8671650271031432,\n      "first_frame_absolute_center_error_mean_px_missing_penalized": 1.7613287685906618,\n      "first_frame_absolute_center_error_p95_px_missing_penalized": 1.7613287685906618,\n      "stability_cutoff": 0.65,\n      "trusted_coverage": 0.9705882352941176,\n      "mean_trusted_center_error_frame_diagonals": 0.009255556768232706,\n      "mean_stability_heuristic": 0.7091014205256876,\n      "mean_valid_threshold_votes_out_of_9": 9,\n      "anchor_relative_trajectory_error_mean_abs_control_grid_units_xy": [\n        0.027758783061847394,\n        0.02021115563025102\n      ],\n      "anchor_relative_trajectory_error_p95_abs_control_grid_units_xy": [\n        0.07842584714828088,\n        0.05025560077710031\n      ],\n      "anchor_relative_trajectory_error_mean_l2_control_grid_units": 0.03794245704049612,\n      "controller_keyframe_count": 10,\n      "controller_keyframe_anchor_relative_mean_abs_control_grid_units_xy": [\n        0.014852642090648849,\n        0.015473451808995231\n      ],\n      "controller_keyframe_anchor_relative_p95_abs_control_grid_units_xy": [\n        0.05091929611650479,\n        0.04836796663606531\n      ],\n      "controller_keyframe_anchor_relative_mean_l2_control_grid_units": 0.024335191869654268\n    }\n  },\n  "per_frame": [\n    {\n      "direction": "left",\n      "frame_index": 0,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7095348340250252,\n      "truth_mask_center_xy_normalized": [\n        0.496875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.5020132211538462,\n        0.4785657051282051\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009439543493252996,\n      "center_error_px_missing_penalized": 1.7613287685906618,\n      "center_error_components_px_xy": [\n        0.822115384615385,\n        -1.5576923076923102\n      ],\n      "anchor_relative_error_tokens_xy": [\n        0.0,\n        0.0\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "left",\n      "frame_index": 1,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7413568788233176,\n      "truth_mask_center_xy_normalized": [\n        0.484375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.4889880952380953,\n        0.47951388888888885\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.008799583272783087,\n      "center_error_px_missing_penalized": 1.6419182962650738,\n      "center_error_components_px_xy": [\n        0.7380952380952444,\n        -1.4666666666666721\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.027306547619045718,\n        0.028445512820511887\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 2,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7257026057726175,\n      "truth_mask_center_xy_normalized": [\n        0.471875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.47656976744186047,\n        0.47911821705426355\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009012955117172577,\n      "center_error_px_missing_penalized": 1.6817314469962588,\n      "center_error_components_px_xy": [\n        0.751162790697677,\n        -1.5046511627907009\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.02305959302325511,\n        0.016575357781752897\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 3,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6999081527792246,\n      "truth_mask_center_xy_normalized": [\n        0.459375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.4637319711538462,\n        0.4784655448717949\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009193124375498518,\n      "center_error_px_missing_penalized": 1.7153493118995715,\n      "center_error_components_px_xy": [\n        0.6971153846153921,\n        -1.5673076923076934\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.04062499999999769,\n        -0.0030048076923072653\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 4,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7677565737032586,\n      "truth_mask_center_xy_normalized": [\n        0.446875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.45112257281553403,\n        0.4783193779904306\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009224445653168609,\n      "center_error_px_missing_penalized": 1.7211935635277118,\n      "center_error_components_px_xy": [\n        0.679611650485441,\n        -1.5813397129186626\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.0463137135922318,\n        -0.00738981413323514\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "left",\n      "frame_index": 5,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7159771553244285,\n      "truth_mask_center_xy_normalized": [\n        0.434375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.4379988262910798,\n        0.478335289514867\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009018957265538129,\n      "center_error_px_missing_penalized": 1.682851390624587,\n      "center_error_components_px_xy": [\n        0.5798122065727629,\n        -1.5798122065727718\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.0787485328638522,\n        -0.006912468400144256\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 6,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7076749985835675,\n      "truth_mask_center_xy_normalized": [\n        0.421875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.4267017326732673,\n        0.4798272357723577\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.008741115549002733,\n      "center_error_px_missing_penalized": 1.6310087767526147,\n      "center_error_components_px_xy": [\n        0.7722772277227641,\n        -1.436585365853663\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.016197400990101807,\n        0.03784591932457726\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 7,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.738124618783184,\n      "truth_mask_center_xy_normalized": [\n        0.409375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.4132335680751174,\n        0.47964669738863286\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.008465404579559263,\n      "center_error_px_missing_penalized": 1.5795637399620213,\n      "center_error_components_px_xy": [\n        0.617370892018787,\n        -1.4539170506912473\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.06654196009389435,\n        0.03242976781283213\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 8,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.705635372428885,\n      "truth_mask_center_xy_normalized": [\n        0.396875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.40234375,\n        0.4786157852564103\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009552660421933914,\n      "center_error_px_missing_penalized": 1.7824353084188567,\n      "center_error_components_px_xy": [\n        0.8750000000000036,\n        -1.5528846153846132\n      ],\n      "anchor_relative_error_tokens_xy": [\n        0.01718750000000102,\n        0.001502403846155298\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "left",\n      "frame_index": 9,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7347092033826895,\n      "truth_mask_center_xy_normalized": [\n        0.384375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.38911830357142857,\n        0.4786729857819905\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009236723509395068,\n      "center_error_px_missing_penalized": 1.7234844943766179,\n      "center_error_components_px_xy": [\n        0.7589285714285676,\n        -1.5473933649289116\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.020535714285715656,\n        0.0032184196135620624\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 10,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6380963935154782,\n      "truth_mask_center_xy_normalized": [\n        0.371875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.3763095238095238,\n        0.4773313492063492\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009754922147858386,\n      "center_error_px_missing_penalized": 1.8201754170279438,\n      "center_error_components_px_xy": [\n        0.7095238095238088,\n        -1.676190476190481\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.03659226190476228,\n        -0.03703067765567836\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 11,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7008346442296005,\n      "truth_mask_center_xy_normalized": [\n        0.359375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.36377369668246445,\n        0.47788309636650866\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009481888399293875,\n      "center_error_px_missing_penalized": 1.7692299241143774,\n      "center_error_components_px_xy": [\n        0.7037914691943126,\n        -1.62322274881517\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.03845527251184855,\n        -0.020478262850893714\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 12,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7222208623332121,\n      "truth_mask_center_xy_normalized": [\n        0.346875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.35096153846153844,\n        0.4768129006410256\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009891498377518798,\n      "center_error_px_missing_penalized": 1.8456592386320785,\n      "center_error_components_px_xy": [\n        0.6538461538461515,\n        -1.7259615384615437\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.05468750000000089,\n        -0.05258413461538547\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "left",\n      "frame_index": 13,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6578005322722802,\n      "truth_mask_center_xy_normalized": [\n        0.334375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.3394318181818182,\n        0.4788259345794392\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009288555043121472,\n      "center_error_px_missing_penalized": 1.7331557641300488,\n      "center_error_components_px_xy": [\n        0.8090909090909193,\n        -1.532710280373836\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.004232954545451362,\n        0.00780688353702319\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 14,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7244080191397543,\n      "truth_mask_center_xy_normalized": [\n        0.321875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.32630681818181817,\n        0.477586887835703\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.00963306963708369,\n      "center_error_px_missing_penalized": 1.797438900912956,\n      "center_error_components_px_xy": [\n        0.7090909090909037,\n        -1.6516587677725116\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.03673295454545644,\n        -0.029364518775062964\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 15,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.705721189695788,\n      "truth_mask_center_xy_normalized": [\n        0.309375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.3144607843137255,\n        0.4778390522875817\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009751546273641495,\n      "center_error_px_missing_penalized": 1.8195455111027683,\n      "center_error_components_px_xy": [\n        0.8137254901960755,\n        -1.6274509803921564\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.0027267156862755826,\n        -0.021799585218701956\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "left",\n      "frame_index": 16,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7525764656420669,\n      "truth_mask_center_xy_normalized": [\n        0.296875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.3017605633802817,\n        0.47921557120500785\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.00904279203864979,\n      "center_error_px_missing_penalized": 1.687298731918624,\n      "center_error_components_px_xy": [\n        0.7816901408450683,\n        -1.495305164319248\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.013138204225352945,\n        0.019495982304081894\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "right",\n      "frame_index": 0,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7095348340250252,\n      "truth_mask_center_xy_normalized": [\n        0.496875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.5020132211538462,\n        0.4785657051282051\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009439543493252996,\n      "center_error_px_missing_penalized": 1.7613287685906618,\n      "center_error_components_px_xy": [\n        0.822115384615385,\n        -1.5576923076923102\n      ],\n      "anchor_relative_error_tokens_xy": [\n        0.0,\n        0.0\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "right",\n      "frame_index": 1,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7773613904644783,\n      "truth_mask_center_xy_normalized": [\n        0.509375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.513888888888889,\n        0.4801311728395062\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.00847791340837503,\n      "center_error_px_missing_penalized": 1.5818977680927604,\n      "center_error_components_px_xy": [\n        0.7222222222222285,\n        -1.4074074074074083\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.03246527777777586,\n        0.04696403133903182\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 2,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6754030589699789,\n      "truth_mask_center_xy_normalized": [\n        0.521875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.527191943127962,\n        0.4794135071090047\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.00913161481125385,\n      "center_error_px_missing_penalized": 1.7038722139736977,\n      "center_error_components_px_xy": [\n        0.8507109004739277,\n        -1.4763033175355496\n      ],\n      "anchor_relative_error_tokens_xy": [\n        0.009293542654026377,\n        0.025434059423987687\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 3,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7160506494420211,\n      "truth_mask_center_xy_normalized": [\n        0.534375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.5392945544554456,\n        0.47690217391304346\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.010124745399616372,\n      "center_error_px_missing_penalized": 1.8891809079270185,\n      "center_error_components_px_xy": [\n        0.78712871287129,\n        -1.7173913043478297\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.011370668316830868,\n        -0.049905936454849864\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 4,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6774009799927323,\n      "truth_mask_center_xy_normalized": [\n        0.546875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.5519871794871796,\n        0.4771252093802345\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.010091191405216735,\n      "center_error_px_missing_penalized": 1.882920052655844,\n      "center_error_components_px_xy": [\n        0.817948717948731,\n        -1.6959798994974893\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.0013541666666625485,\n        -0.04321487243911848\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "right",\n      "frame_index": 5,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7385458740783271,\n      "truth_mask_center_xy_normalized": [\n        0.559375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.5627053990610329,\n        0.478971048513302\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.008626083182511003,\n      "center_error_px_missing_penalized": 1.6095448345009828,\n      "center_error_components_px_xy": [\n        0.5328638497652705,\n        -1.5187793427230076\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.09400674882628723,\n        0.012160301552907038\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 6,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7026671926451975,\n      "truth_mask_center_xy_normalized": [\n        0.571875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.5770449308755761,\n        0.4788177830940989\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009337919592848958,\n      "center_error_px_missing_penalized": 1.742366718202742,\n      "center_error_components_px_xy": [\n        0.8271889400921673,\n        -1.5334928229665081\n      ],\n      "anchor_relative_error_tokens_xy": [\n        0.0016489055299542255,\n        0.007562338976813132\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 7,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6626170121373872,\n      "truth_mask_center_xy_normalized": [\n        0.584375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.5886443661971831,\n        0.4781396713615023\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009316790222859491,\n      "center_error_px_missing_penalized": 1.7384241793234791,\n      "center_error_components_px_xy": [\n        0.683098591549296,\n        -1.5985915492957794\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.04518045774647894,\n        -0.01278101300108414\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 8,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6743338305297055,\n      "truth_mask_center_xy_normalized": [\n        0.596875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.6019406392694064,\n        0.47901642628205127\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009205563236315654,\n      "center_error_px_missing_penalized": 1.717670284669216,\n      "center_error_components_px_xy": [\n        0.8105022831050235,\n        -1.5144230769230802\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.0037742579908675022,\n        0.01352163461538436\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "right",\n      "frame_index": 9,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6510130328115478,\n      "truth_mask_center_xy_normalized": [\n        0.609375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.6142772511848341,\n        0.47911729857819907\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009094235410307575,\n      "center_error_px_missing_penalized": 1.69689757433287,\n      "center_error_components_px_xy": [\n        0.7843601895734587,\n        -1.5047393364928912\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.01227043838862607,\n        0.016547803499818436\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 10,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7157114484959073,\n      "truth_mask_center_xy_normalized": [\n        0.621875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.6263574660633484,\n        0.47897376543209874\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009000272128934375,\n      "center_error_px_missing_penalized": 1.679364922378664,\n      "center_error_components_px_xy": [\n        0.7171945701357529,\n        -1.5185185185185226\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.03409926470588043,\n        0.01224180911680861\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 11,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6857624461958536,\n      "truth_mask_center_xy_normalized": [\n        0.634375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.6380083732057417,\n        0.47787081339712917\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009246418224861925,\n      "center_error_px_missing_penalized": 1.7252934358013046,\n      "center_error_components_px_xy": [\n        0.5813397129186626,\n        -1.6244019138756016\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.07825209330143479,\n        -0.020846751932278584\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 12,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7293674599760658,\n      "truth_mask_center_xy_normalized": [\n        0.646875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.6518947963800905,\n        0.4790702160493827\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.00916263990029546,\n      "center_error_px_missing_penalized": 1.709661199629215,\n      "center_error_components_px_xy": [\n        0.8031674208144857,\n        -1.5092592592592613\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.006158088235292292,\n        0.015135327635327767\n      ],\n      "controller_keyframe": true\n    },\n    {\n      "direction": "right",\n      "frame_index": 13,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7144628490771807,\n      "truth_mask_center_xy_normalized": [\n        0.659375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.6640169902912622,\n        0.47852790880503143\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009266146836486876,\n      "center_error_px_missing_penalized": 1.728974606532094,\n      "center_error_components_px_xy": [\n        0.7427184466019376,\n        -1.5613207547169843\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.0258040048543704,\n        -0.0011338896952106658\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 14,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7232231713455315,\n      "truth_mask_center_xy_normalized": [\n        0.671875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.6766203703703704,\n        0.4796006944444444\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.008811513578178939,\n      "center_error_px_missing_penalized": 1.6441443774444027,\n      "center_error_components_px_xy": [\n        0.7592592592592595,\n        -1.4583333333333375\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.02042824074074079,\n        0.03104967948717896\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 15,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.6889277016488766,\n      "truth_mask_center_xy_normalized": [\n        0.684375,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.688769035532995,\n        0.4768688725490196\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009961289261410374,\n      "center_error_px_missing_penalized": 1.858681551805535,\n      "center_error_components_px_xy": [\n        0.7030456852792,\n        -1.7205882352941195\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.03869765228426014,\n        -0.05090497737556543\n      ],\n      "controller_keyframe": false\n    },\n    {\n      "direction": "right",\n      "frame_index": 16,\n      "scoreable": true,\n      "valid_threshold_count": 9,\n      "stability_heuristic": 0.7190268656031841,\n      "truth_mask_center_xy_normalized": [\n        0.696875,\n        0.4947916666666667\n      ],\n      "observed_center_xy_normalized": [\n        0.7018995098039216,\n        0.4786287167449139\n      ],\n      "center_error_frame_diagonals_missing_penalized": 0.009365630252339706,\n      "center_error_px_missing_penalized": 1.747537262921622,\n      "center_error_components_px_xy": [\n        0.8039215686274481,\n        -1.5516431924882657\n      ],\n      "anchor_relative_error_tokens_xy": [\n        -0.005912990196079493,\n        0.0018903485012639054\n      ],\n      "controller_keyframe": true\n    }\n  ],\n  "input_sha256": {\n    "kubric_gate0_fixture_v1/left/frames/frame_000.png": "d07ccdb3596697918d2d30ed944e854c97c85210d735ca360e56d2754431c3a1",\n    "kubric_gate0_fixture_v1/left/masks/mask_000.png": "ef0098f77440ece3bb05f8e6845abf58174fcb9af86b53426d6ad9997864fad0",\n    "kubric_gate0_fixture_v1/left/frames/frame_001.png": "26a5fc06331a70f7c9597f99ec4485e82773f8922e3824bbf5e984175ea8e1d8",\n    "kubric_gate0_fixture_v1/left/masks/mask_001.png": "80f8fe908a31786d3ebb7ece2a7ae6f650033e7d6b38623e040a3917c03e8d47",\n    "kubric_gate0_fixture_v1/left/frames/frame_002.png": "891a9745debdc476f93a2e385b1dd6aa5b271ca551f84ddec0231df486d95e8a",\n    "kubric_gate0_fixture_v1/left/masks/mask_002.png": "a11c3e40a1d3fd866305c2cbae0450f98a530031e1341d7dd0d6e5e15c87e4a1",\n    "kubric_gate0_fixture_v1/left/frames/frame_003.png": "7364511a0dee64ba5dc124037b3eb8fbb7a3794fa7fbee9b3b21edf0b2fe963d",\n    "kubric_gate0_fixture_v1/left/masks/mask_003.png": "f9d0df6d7d95dd17d14aa486636243d5138135caaf8c1935a8fc7bca8c4df037",\n    "kubric_gate0_fixture_v1/left/frames/frame_004.png": "47355df6e8cecbe49af0be67caf4511206b62f73000ee7ffa89f49931814ed68",\n    "kubric_gate0_fixture_v1/left/masks/mask_004.png": "d4ff0dd04b0e21474be0aef2a211dac7ea7951a991de3b22a1165e6e71194e04",\n    "kubric_gate0_fixture_v1/left/frames/frame_005.png": "5927c05fe27ded7ce6897a0b91ac0072b6404f73a2ffa74239498f76baf0a620",\n    "kubric_gate0_fixture_v1/left/masks/mask_005.png": "5a8e79d221296bb53d584edc838b39ef911547cde000cd2b21fa0707fd1db9a8",\n    "kubric_gate0_fixture_v1/left/frames/frame_006.png": "2da34810adeaf03e906a1fdde730e491cbf4ff955fcc8b5a5c56b1dc0c5611d6",\n    "kubric_gate0_fixture_v1/left/masks/mask_006.png": "5965bdc9caf4cf45b1023ce8782ca3edc442d6a0bcdb82d2daf79c2b654c481f",\n    "kubric_gate0_fixture_v1/left/frames/frame_007.png": "bb59bff4f290fd45b2da74b70df0b18116f7687fd2eee8dce58336f1dc935442",\n    "kubric_gate0_fixture_v1/left/masks/mask_007.png": "237d0afe26be1279846b80113adca576b76f001c072315ee9e02c97cbee47e62",\n    "kubric_gate0_fixture_v1/left/frames/frame_008.png": "b09232a676b949fe7bd9d8eeef70260476081fb1626413298781efc0a183ad06",\n    "kubric_gate0_fixture_v1/left/masks/mask_008.png": "f7a5d3ca6c716abe22070db4fc7c1ba0e0838ed0f61ae108b4ed1c2bfda432e8",\n    "kubric_gate0_fixture_v1/left/frames/frame_009.png": "212bf9e05729bd7e8d45fc8103ac1c7f88284a194568524f22a1d467443308f4",\n    "kubric_gate0_fixture_v1/left/masks/mask_009.png": "aaa9f48875105608fd7a7d848409d0281061480c6fd849886c71fd257ecd5ade",\n    "kubric_gate0_fixture_v1/left/frames/frame_010.png": "c975bf70583488912c4db6b5debc945e6075cb635f21167305c69484299129ab",\n    "kubric_gate0_fixture_v1/left/masks/mask_010.png": "3a78aef2261f25a6247c138d6f8c6c4ff3f178511579cc60fe2250ea225517fd",\n    "kubric_gate0_fixture_v1/left/frames/frame_011.png": "c97b58b37e638dea4fd4729229dfca30bdaa1cec9062cf430b36a4f59f824f3f",\n    "kubric_gate0_fixture_v1/left/masks/mask_011.png": "cfea83d8f987abb686b59e8f3066dda1f5927e0a85dc269b28c10de527f8824d",\n    "kubric_gate0_fixture_v1/left/frames/frame_012.png": "c24b1cd29621482d96d876b989cb7a1ad053c473cfb6df921de98bb5c57bbde5",\n    "kubric_gate0_fixture_v1/left/masks/mask_012.png": "3c42222704078f636060c21dbfca65609057993eb2c0f029fa1318d2a780d45c",\n    "kubric_gate0_fixture_v1/left/frames/frame_013.png": "96a3b688372a8bdb42125f361449a3d1f7f72279f235a885524daa5a315a33ce",\n    "kubric_gate0_fixture_v1/left/masks/mask_013.png": "a9bba7b4224d45745efa0adac0a8a378365c8a8b63378a9d30741ede6cb38acd",\n    "kubric_gate0_fixture_v1/left/frames/frame_014.png": "bccb2eb2c60f8592c5135b23310bd950c980fefaec09a98784f6f888d2b47ba0",\n    "kubric_gate0_fixture_v1/left/masks/mask_014.png": "d8347c74263f07fb412b46d5b2a521cde566d1e3f4bccb0155b5ab0ff27a8ffb",\n    "kubric_gate0_fixture_v1/left/frames/frame_015.png": "af4178a655246fc4e190d0051cb4020dfafc06416c8ced86eb53250a28fb9b6e",\n    "kubric_gate0_fixture_v1/left/masks/mask_015.png": "03b3f738dcc0f45696ebde62586619a8636eeef9ca9ce514459096d289472f99",\n    "kubric_gate0_fixture_v1/left/frames/frame_016.png": "9512d10da88179a97d6c4a18b0c79833a04dc9657929763ec6a92b3531eaa703",\n    "kubric_gate0_fixture_v1/left/masks/mask_016.png": "becd20c8695de954dc8ca5c964c12cd8f027b9feaa7bca510a4c8eaf809f0c0e",\n    "kubric_gate0_fixture_v1/right/frames/frame_000.png": "d07ccdb3596697918d2d30ed944e854c97c85210d735ca360e56d2754431c3a1",\n    "kubric_gate0_fixture_v1/right/masks/mask_000.png": "ef0098f77440ece3bb05f8e6845abf58174fcb9af86b53426d6ad9997864fad0",\n    "kubric_gate0_fixture_v1/right/frames/frame_001.png": "c0c037d15bd34693e851fbf1bfc6492df91b2bc97e6b441edd6bde1ef9918422",\n    "kubric_gate0_fixture_v1/right/masks/mask_001.png": "839d1d8f06cae56fc4d52599b8cac1be5e3b7419dc0d5e0ad2e90e10f23a5bcf",\n    "kubric_gate0_fixture_v1/right/frames/frame_002.png": "7c40cdc0bcdd85a7d6956b4db4d0bd04607faa33b6861d33198feede03a0d75f",\n    "kubric_gate0_fixture_v1/right/masks/mask_002.png": "54a9b52f4679dc9ddf8f7f969126d4dff411b728990158657610e59e2354721d",\n    "kubric_gate0_fixture_v1/right/frames/frame_003.png": "08026b91b45d42c0e26fd9b7d0427db7bfefd4ac381808b31cc2223ddc13d39b",\n    "kubric_gate0_fixture_v1/right/masks/mask_003.png": "8fb88cb2d529e89ff462cbefe66f745254b6160dd210d15b882dda35c2d0be21",\n    "kubric_gate0_fixture_v1/right/frames/frame_004.png": "f54a93b49bb14796dac0cf0bd9732c6fab9fb9f09f7d37d21e5963893b14d549",\n    "kubric_gate0_fixture_v1/right/masks/mask_004.png": "3a7a4d959a8f5d1cd45c8e58f791314b6a5ef1c4fe341f867e775b0f0c10b1c9",\n    "kubric_gate0_fixture_v1/right/frames/frame_005.png": "dfe321c1b4a2ea95bbd7b9304ac66a4235defe055e7df593f9b1b865437bbf40",\n    "kubric_gate0_fixture_v1/right/masks/mask_005.png": "ac28c8f9600d8cff13f9031346ceae90e9f55f4ad8e7279983f15a3db0087726",\n    "kubric_gate0_fixture_v1/right/frames/frame_006.png": "a06179f5dd2e41de05495ca0aeae746b654bbf5382ac14be6a3d11991b042c30",\n    "kubric_gate0_fixture_v1/right/masks/mask_006.png": "ed6330b9a8828ed98daff35826227c66719fa1069baad1d01d2cd82ac019daf3",\n    "kubric_gate0_fixture_v1/right/frames/frame_007.png": "4307e6797f74b10dd5c7e6c6da88dcb9500869009f65c043f163d05ece609c6d",\n    "kubric_gate0_fixture_v1/right/masks/mask_007.png": "972f3646be46db9b047f3c872db78b0bcf74a74c496095a1f73cb950c3fde9d0",\n    "kubric_gate0_fixture_v1/right/frames/frame_008.png": "4c137c07d94f9c863f1346ba8878d23d0fe7c0131c06fd64cd051ede253b4036",\n    "kubric_gate0_fixture_v1/right/masks/mask_008.png": "010069c178283c999b484186d1687e0b53c76fda58bb0aa684083cf7342e06af",\n    "kubric_gate0_fixture_v1/right/frames/frame_009.png": "308728007deef482b580577eeb9bbf3c23c20d95819d5e37932149f61f69bb1c",\n    "kubric_gate0_fixture_v1/right/masks/mask_009.png": "900c22337581bc6ee30fa600e73bce79cb14e80f990375acd2ea713d8af63e17",\n    "kubric_gate0_fixture_v1/right/frames/frame_010.png": "ed96e9839dc941d9517756b13463ed2cfff4bcd88c0f667a9abd3d67dbe57e6b",\n    "kubric_gate0_fixture_v1/right/masks/mask_010.png": "e95d899ee234602710db148d835b1ed927e9920c93720b23b6edeae2e89851c2",\n    "kubric_gate0_fixture_v1/right/frames/frame_011.png": "ff3e46b03a215ff01ed61e10bc0d7453d6e8d9af830d744d57e9b2363d3529af",\n    "kubric_gate0_fixture_v1/right/masks/mask_011.png": "08d48943878e797c5e15027c41de9f86c3d84c30728a383dfaf87f1f3be371b1",\n    "kubric_gate0_fixture_v1/right/frames/frame_012.png": "97bccb2818fd8a81828c735183d1e70945386c749f0c990beba9e12b62251151",\n    "kubric_gate0_fixture_v1/right/masks/mask_012.png": "7db88053c0ffad21c428290f2d38951c5fafe466783d97a0255cb64a492fb28c",\n    "kubric_gate0_fixture_v1/right/frames/frame_013.png": "0aac819d47f6502db55b11c2ecad1c387b749541d15af53673da68f9ccd71b63",\n    "kubric_gate0_fixture_v1/right/masks/mask_013.png": "74ef1654f75b2a599c8c04048a88b17b4a6c966d957de1f9acbbe4618982ce01",\n    "kubric_gate0_fixture_v1/right/frames/frame_014.png": "5c57c45aaaf9693d4560f3b912b8655dc186cf01f2052fb8c95bed6c72114377",\n    "kubric_gate0_fixture_v1/right/masks/mask_014.png": "5d7a3b08e83adc4eb77c5fc6a3249b78f01508d4f0882bbcec598cc574321b26",\n    "kubric_gate0_fixture_v1/right/frames/frame_015.png": "ad4d5812b91b34ff8303ff739269347d60d18c7a7b9d386fc7fb42dc5c04331b",\n    "kubric_gate0_fixture_v1/right/masks/mask_015.png": "055e4be239aa5cb2dd459f9637d2d136edf5795730b723e9411062b4e4838bd1",\n    "kubric_gate0_fixture_v1/right/frames/frame_016.png": "f7cb28a0207fa0e23104aef8bb945a82bad9a11e46f410799ae880f6dd01a657",\n    "kubric_gate0_fixture_v1/right/masks/mask_016.png": "2d7a897087299787892cf458c51daab3318312e2a40be11f0b8c1f4e598dd9e5"\n  },\n  "limitations": [\n    "This tests a red Kubric sphere on its controlled background, not generated Wan previews or other object classes.",\n    "Threshold agreement is not a calibrated identity probability; stability cutoff 0.65 is provisional and fixture-specific.",\n    "The 0.65 cutoff and its coverage were assessed on these same two development paths; no held-out path or generated preview validates it yet.",\n    "The first-frame anchor cancels a fixed observer offset for relative path measurements; absolute first-frame localization is reported separately because the start box is locked and cannot be corrected.",\n    "The 34 synthetic frames are correlated across two trajectories and do not replace held-out generated samples or human labels.",\n    "No controller was run against model output; this is observer localization evidence only."\n  ]\n}\n'
observer_calibration_path = Path("/kaggle/working/DiTraj/red_observer_kubric_calibration_v1.json")
observer_calibration_path.write_text(observer_calibration_source)
print("Wrote known-truth red-observer calibration:", observer_calibration_path)

unipc_cpu_smoke_source = '#!/usr/bin/env python3\n"""Run the adapter capture against actual Diffusers 0.33.1 UniPC on CPU tensors.\n\nFrom this workspace, this can use an isolated package install without changing\nthe host environment:\n\n    PYTHONPATH=/tmp/latentgate-diffusers-0.33.1 python -m research.smoke_pinned_unipc_capture\n"""\n\nfrom __future__ import annotations\n\nimport importlib.metadata\nimport json\nfrom pathlib import Path\n\nimport torch\n\ntry:\n    from .ditraj_feedback_adapter import UniPCX0Capture\nexcept ImportError:  # flat import after notebook staging into DiTraj\'s root\n    from ditraj_feedback_adapter import UniPCX0Capture\n\n\ndef run() -> dict[str, object]:\n    version = importlib.metadata.version("diffusers")\n    if version != "0.33.1":\n        raise RuntimeError(f"Expected Diffusers 0.33.1, found {version}")\n    from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler\n\n    torch.manual_seed(2026)\n    scheduler = UniPCMultistepScheduler(\n        prediction_type="flow_prediction",\n        use_flow_sigmas=True,\n        num_train_timesteps=1000,\n        flow_shift=3.0,\n    )\n    if scheduler.config.thresholding:\n        raise RuntimeError("Smoke expects thresholding=False so the direct conversion formula is checkable")\n    scheduler.set_timesteps(8, device="cpu")\n    sample = torch.randn((1, 2, 5, 4, 6), dtype=torch.float32)\n    result: dict[str, object] | None = None\n    for step, timestep in enumerate(scheduler.timesteps):\n        model_output = torch.randn_like(sample)\n        capture = None\n        expected_x0 = None\n        if step == 4:\n            capture = UniPCX0Capture(scheduler, capture_step=step)\n            capture.begin(step, timestep)\n            sigma = scheduler.sigmas[scheduler.step_index].to(sample.device)\n            expected_x0 = sample - sigma * model_output\n        try:\n            sample = scheduler.step(model_output, timestep, sample, return_dict=False)[0]\n            if capture is not None:\n                packet = capture.consume(step)\n                assert expected_x0 is not None\n                torch.testing.assert_close(packet["x0_hat"], expected_x0, rtol=1e-6, atol=1e-6)\n                result = {\n                    "experiment_id": "EXP-DITRAJ-UNIPC-EXACT-CAPTURE-CPU-001",\n                    "status": "passed",\n                    "diffusers_version": version,\n                    "device": "cpu",\n                    "seed": 2026,\n                    "num_inference_steps": 8,\n                    "captured_step_zero_based": packet["step_zero_based"],\n                    "scheduler_step_index": packet["scheduler_step_index"],\n                    "captured_tensor_shape": packet["shape"],\n                    "captured_tensor_dtype": packet["dtype"],\n                    "max_abs_error_vs_flow_prediction_conversion": float(\n                        (packet["x0_hat"] - expected_x0).abs().max().item()\n                    ),\n                    "scheduler_index_after_update": int(scheduler.step_index),\n                    "final_sample_finite": bool(torch.isfinite(sample).all().item()),\n                }\n        finally:\n            if capture is not None:\n                capture.close()\n    if result is None or result["final_sample_finite"] is not True:\n        raise RuntimeError("Pinned UniPC CPU smoke did not capture a finite step-4 estimate")\n    return result\n\n\ndef main() -> None:\n    result = run()\n    output = Path(__file__).with_name("ditraj_unipc_capture_smoke_v1.json")\n    output.write_text(json.dumps(result, indent=2) + "\\n")\n    print(json.dumps({"output": str(output), **result}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'
unipc_cpu_smoke_result = '{\n  "experiment_id": "EXP-DITRAJ-UNIPC-EXACT-CAPTURE-CPU-001",\n  "status": "passed",\n  "diffusers_version": "0.33.1",\n  "device": "cpu",\n  "seed": 2026,\n  "num_inference_steps": 8,\n  "captured_step_zero_based": 4,\n  "scheduler_step_index": 4,\n  "captured_tensor_shape": [\n    1,\n    2,\n    5,\n    4,\n    6\n  ],\n  "captured_tensor_dtype": "torch.float32",\n  "max_abs_error_vs_flow_prediction_conversion": 0.0,\n  "scheduler_index_after_update": 5,\n  "final_sample_finite": true\n}\n'
for filename, content in {
    "smoke_pinned_unipc_capture.py": unipc_cpu_smoke_source,
    "ditraj_unipc_capture_smoke_v1.json": unipc_cpu_smoke_result,
}.items():
    target = Path("/kaggle/working/DiTraj") / filename
    target.write_text(content)
    print("Wrote", target)


In [ ]:
# Tiny CPU-only check of exact step-4 capture against pinned Diffusers 0.33.1.
%cd /kaggle/working/DiTraj
!python smoke_pinned_unipc_capture.py


In [ ]:
# A smoke run needs its own fresh receipt; the CLI validates it again before imports/CUDA.
%cd /kaggle/working/DiTraj
run_receipt = collect_ditraj_receipt("paired_path_steps_1", PAIRED_RUNNER_SHA256)
!python -u run_ditraj_paired_path_pilot.py --num-steps 1 --output-dir /kaggle/working/ditraj_paired_path_smoke --admission-receipt /kaggle/working/ditraj_admission_receipt.json --notebook-sha256 $ADMITTED_NOTEBOOK_SHA256


In [ ]:
# Preserve the registered six arms; the long screen is still disabled by default.
import subprocess
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    run_receipt = collect_ditraj_receipt("paired_path_steps_30", PAIRED_RUNNER_SHA256)
    subprocess.run([
        "python", "-u", "run_ditraj_paired_path_pilot.py",
        "--num-steps", "30",
        "--output-dir", "/kaggle/working/ditraj_paired_path_pilot",
        "--admission-receipt", "/kaggle/working/ditraj_admission_receipt.json",
        "--notebook-sha256", ADMITTED_NOTEBOOK_SHA256,
    ], check=True)
else:
    print("Full 30-step experiment remains disabled; review the one-step smoke first.")


In [ ]:
# Review diagnostics only. The runner reports this packet as non-scoreable/non-exportable.
from IPython.display import display, Image
from pathlib import Path
import json
smoke_dir = Path("/kaggle/working/ditraj_paired_path_smoke")
report_path = smoke_dir / "pilot_report.json"
if report_path.exists():
    display(Image(filename=str(smoke_dir / "contact_sheet.png")))
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2))
    packet = report.get("primary_evaluation", {})
    assert packet.get("scoreable") is False and packet.get("exportable") is False
    print("Local diagnostic only: do not archive, export, or send frames to raters.")
else:
    print("Run the gated one-step smoke before reviewing local diagnostics.")


## Later stage: step-4 latent feedback smoke

Run this only after Gate 1 human-reference reliability, the generated-preview observer gate, and actuator identification have passed; the complete open-loop DiTraj screen must also be finished and reviewed, and a fresh check must confirm free T4 quota. This is a one-direction, eight-step mechanism smoke with three matched arms: full DiTraj baseline, a VAE-decode-matched zero-update control, and bounded x0-hat feedback. It checks code path, mask change, time, and memory; it is not an efficacy result.

The script captures UniPC’s exact converted x0 estimate after step 4, decodes all 17 frames, observes the full frames with the red-threshold ensemble, then updates the five latent-time foreground boxes for the remaining eligible guidance steps.


In [ ]:
# Keep disabled until all registered observer/actuator/open-loop gates pass.
import subprocess
RUN_FEEDBACK_SMOKE = False
if RUN_FEEDBACK_SMOKE:
    feedback_arm = "feedback_path_right_steps_8"
    feedback_receipt = collect_ditraj_receipt(feedback_arm, FEEDBACK_RUNNER_SHA256)
    subprocess.run([
        "python", "-u", "run_ditraj_feedback_pilot.py",
        "--num-steps", "8", "--path", "right",
        "--output-dir", "/kaggle/working/ditraj_feedback_smoke",
        "--admission-receipt", feedback_receipt,
        "--notebook-sha256", ADMITTED_NOTEBOOK_SHA256,
    ], cwd="/kaggle/working/DiTraj", check=True)
else:
    print("Feedback smoke remains disabled; complete and review its prerequisite gates first.")


In [ ]:
from IPython.display import display, Image
from pathlib import Path
import json
if globals().get("RUN_FEEDBACK_SMOKE", False):
    feedback_dir = Path("/kaggle/working/ditraj_feedback_smoke")
    feedback_report = json.loads((feedback_dir / "feedback_smoke_report.json").read_text())
    for arm, result in feedback_report["arms"].items():
        print(arm, "seconds=", round(result["generation_seconds"], 1), "peak_VRAM_GiB=", round(result["peak_vram_allocated_bytes"] / 1024**3, 2), "proxy_coverage=", result["secondary_proxy_metrics"]["valid_frame_coverage"])
    display(Image(filename=str(feedback_dir / "contact_sheet.png")))
    if feedback_report["arms"]["x0_hat_feedback"]["feedback_report"]:
        print(json.dumps(feedback_report["arms"]["x0_hat_feedback"]["feedback_report"]["rasterization"], indent=2))
else:
    print("No feedback-smoke artifacts to review; its run cell is disabled by default.")
